<a href="https://colab.research.google.com/github/KingExecutioner/Projects/blob/main/Loan_default_dataset%E2%80%94_Batch_Ingestion%2C_ML_%26_Observability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Loan_default_dataset — Batch Ingestion, ML & Observability (Colab)

Simulates a recurring batch pipeline on top of the credit risk / churn model:
new "batches" land each run, get appended to a growing data lake, validated
for schema drift, checked for statistical data drift, feature-engineered,
and periodically used to retrain a model — all logged for lineage in SQLite,
with an inline observability dashboard at the end.

**Persistence**: Colab runtimes are ephemeral — anything written to `/content`
disappears when the runtime recycles. This notebook mounts Google Drive and
stores the data lake + SQLite metadata store there, so re-running the
simulation later (even in a brand-new runtime) **continues** the batch
history instead of losing it.

**How to use it**
1. Runtime -> Run all, once. This does an initial 10-batch run with `reset=True`.
2. Whenever you want to simulate "the next few days" arriving, re-run just the
   **"Run the simulation"** cell with `reset=False` -- it picks up where it left off.
3. Re-run the **dashboard** cell any time to refresh the charts inline.


## 1. Mount Google Drive & install dependencies

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/credit_risk_pipeline'
DATA_DIR = f'{PROJECT_DIR}/data'
BATCHES_DIR = f'{DATA_DIR}/batches'
LAKE_PATH = f'{DATA_DIR}/credit_risk_lake.csv'
MODEL_PATH = f'{DATA_DIR}/model.joblib'
DB_PATH = f'{DATA_DIR}/metadata.db'
DASHBOARD_PATH = f'{PROJECT_DIR}/dashboard.html'

os.makedirs(BATCHES_DIR, exist_ok=True)
print('Data will persist at:', PROJECT_DIR)


Mounted at /content/drive
Data will persist at: /content/drive/MyDrive/credit_risk_pipeline


In [8]:
!pip install fairlearn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 5.7 MB/s eta 0:00:00


In [9]:
import hashlib, json, sqlite3
from contextlib import contextmanager
from datetime import datetime, timedelta
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from fairlearn.metrics import MetricFrame, demographic_parity_difference


## 2. Expected schema & schema-drift detection

In [23]:
import json
from dataclasses import dataclass, field

# dtype "kinds" we care about at the raw-ingestion layer
KIND_MAP = {
    "int64": "numeric",
    "float64": "numeric",
    "bool": "boolean",
    "object": "categorical",
}

EXPECTED_SCHEMA = {
    "LoanID": "categorical",
    "Age": "numeric",
    "Income": "numeric",
    "LoanAmount": "numeric",
    "CreditScore": "numeric",
    "MonthsEmployed": "numeric",
    "NumCreditLines": "numeric",
    "InterestRate": "numeric",
    "LoanTerm": "numeric",
    "DTIRatio": "numeric",
    "Education": "categorical",
    "EmploymentType": "categorical",
    "MaritalStatus": "categorical",
    "HasMortgage": "categorical",
    "HasDependents": "categorical",
    "LoanPurpose": "categorical",
    "HasCoSigner": "categorical",
    "Default": "numeric", # target for new dataset
    "batch_id": "categorical",
    "ingested_at": "categorical",
}


def infer_kind(dtype) -> str:
    return KIND_MAP.get(str(dtype), "categorical")


def snapshot_schema(df) -> dict:
    """Return {column: kind} for a dataframe."""
    return {col: infer_kind(dtype) for col, dtype in df.dtypes.items()}


@dataclass
class SchemaDiff:
    added_columns: list = field(default_factory=list)
    removed_columns: list = field(default_factory=list)
    type_changed: list = field(default_factory=list)  # list of (col, old_kind, new_kind)

    @property
    def has_drift(self) -> bool:
        return bool(self.added_columns or self.removed_columns or self.type_changed)

    def to_dict(self) -> dict:
        return {
            "added_columns": self.added_columns,
            "removed_columns": self.removed_columns,
            "type_changed": self.type_changed,
            "has_drift": self.has_drift,
        }


def diff_schema(expected: dict, observed: dict) -> SchemaDiff:
    added = sorted(set(observed) - set(expected))
    removed = sorted(set(expected) - set(observed))
    changed = []
    for col in sorted(set(expected) & set(observed)):
        if expected[col] != observed[col]:
            changed.append((col, expected[col], observed[col]))
    return SchemaDiff(added_columns=added, removed_columns=removed, type_changed=changed)


### Updated `EXPECTED_SCHEMA` for New Dataset

The `EXPECTED_SCHEMA` dictionary in the previous cell (`88852de5`) has been updated to align with the column names and inferred data types from the `Loan_default_dataset_students.csv` dataset. This ensures that the pipeline's schema validation step correctly interprets the structure of the new incoming data.

**Key changes include:**
-   Replacement of original synthetic data column names (e.g., `SeriousDlqin2yrs`, `RevolvingUtilizationOfUnsecuredLines`) with columns from the new dataset (e.g., `LoanID`, `Age`, `Income`, `LoanAmount`, `CreditScore`, etc.).
-   The target variable has been identified as `Default` for the new dataset.
-   Retention of pipeline-specific columns like `batch_id` and `ingested_at` which are added during the batch generation process.

This schema definition is critical for the `diff_schema` function to detect any discrepancies between the expected data structure and the actual structure of incoming data batches, helping to identify potential schema drift early in the pipeline.

## 3. Synthetic batch generator (drift injected at runs 6 and 9)

### Using `Loan_default_dataset_students.csv` for Batch Generation

The following code replaces the original synthetic batch generation. Instead, it will load `Loan_default_dataset_students.csv` once and then sample from it for each simulated batch. This ensures that the pipeline processes real data, while still maintaining the batching and lineage tracking aspects of the simulation.

**Note**: You must upload `Loan_default_dataset_students.csv` to the `/content/` directory in your Colab environment for this to work. If the file is not found, the system will temporarily revert to a simplified synthetic data generation for demonstration purposes, but you will need to upload the file to use it.

In [11]:
import pandas as pd # Import pandas
import numpy as np # Import numpy for dummy data generation

FULL_DATASET_DF = None # Initialize global variable for the full dataset

def _load_full_dataset():
    global FULL_DATASET_DF
    if FULL_DATASET_DF is None:
        try:
            # Assuming the file is uploaded to /content/ as specified by the user
            FULL_DATASET_DF = pd.read_csv('/content/Loan_default_dataset_students.csv')
            print("Loaded 'Loan_default_dataset_students.csv'. Subsequent batches will be sampled from this dataset.")
        except FileNotFoundError:
            print("ERROR: 'Loan_default_dataset_students.csv' not found. Please upload it to /content/.")
            # Fallback to a dummy DataFrame if the file is not found to prevent immediate crashes
            # This dummy data will likely cause schema mismatches, prompting the user to fix.
            print("Using a dummy DataFrame as fallback. Please upload the specified CSV.")
            # Create a dummy DataFrame with similar columns to the original EXPECTED_SCHEMA to prevent immediate crashes
            # If the original columns are not present, further errors might occur downstream.
            dummy_data = {
                "SeriousDlqin2yrs": np.random.randint(0, 2, 500),
                "RevolvingUtilizationOfUnsecuredLines": np.random.rand(500),
                "age": np.random.randint(21, 76, 500),
                "NumberOfTime30-59DaysPastDueNotWorse": np.random.randint(0, 5, 500),
                "DebtRatio": np.random.rand(500) * 0.5,
                "MonthlyIncome": np.random.normal(5400, 2200, 500).clip(500, None),
                "NumberOfOpenCreditLinesAndLoans": np.random.randint(0, 15, 500),
                "NumberOfTimes90DaysLate": np.random.randint(0, 3, 500),
                "NumberRealEstateLoansOrLines": np.random.randint(0, 4, 500),
                "NumberOfTime60-89DaysPastDueNotWorse": np.random.randint(0, 2, 500),
                "NumberOfDependents": np.random.randint(0, 5, 500),
            }
            FULL_DATASET_DF = pd.DataFrame(dummy_data)
    return FULL_DATASET_DF

In [12]:
print('Loading the dataset to infer its schema for EXPECTED_SCHEMA update...')
df_sample = _load_full_dataset()

# Infer the schema from the loaded dataset
inferred_schema = {col: infer_kind(dtype) for col, dtype in df_sample.dtypes.items()}

print('\nInferred schema from Loan_default_dataset_students.csv:')
for col, kind in inferred_schema.items():
    print(f"  '{col}': '{kind}',")

# Now, the EXPECTED_SCHEMA in the cell '88852de5' should be updated manually or by another modification step.
# I will proceed to update the `generate_batch` function to use this dataset.


Loading the dataset to infer its schema for EXPECTED_SCHEMA update...
Loaded 'Loan_default_dataset_students.csv'. Subsequent batches will be sampled from this dataset.

Inferred schema from Loan_default_dataset_students.csv:
  'LoanID': 'categorical',
  'Age': 'numeric',
  'Income': 'numeric',
  'LoanAmount': 'numeric',
  'CreditScore': 'numeric',
  'MonthsEmployed': 'numeric',
  'NumCreditLines': 'numeric',
  'InterestRate': 'numeric',
  'LoanTerm': 'numeric',
  'DTIRatio': 'numeric',
  'Education': 'categorical',
  'EmploymentType': 'categorical',
  'MaritalStatus': 'categorical',
  'HasMortgage': 'categorical',
  'HasDependents': 'categorical',
  'LoanPurpose': 'categorical',
  'HasCoSigner': 'categorical',
  'Default': 'numeric',


In [13]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

BASE_DATE = datetime(2026, 1, 1)


def generate_batch(run_number: int, n_samples: int = 250, seed_offset: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(1000 + run_number + seed_offset)

    # --- distribution shift after run 9: simulate a macro shock ---
    shocked = run_number >= 9
    income_scale = 0.65 if shocked else 1.0
    delinquency_boost = 1.8 if shocked else 1.0

    data = {
        "SeriousDlqin2yrs": rng.binomial(1, 0.10 * delinquency_boost, n_samples).clip(0, 1),
        "RevolvingUtilizationOfUnsecuredLines": rng.beta(2, 5, n_samples) * (1.6 if shocked else 1.0),
        "age": rng.integers(21, 76, n_samples),
        "NumberOfTime30-59DaysPastDueNotWorse": rng.poisson(0.3 * delinquency_boost, n_samples),
        "DebtRatio": rng.exponential(0.4, n_samples) * (1.3 if shocked else 1.0),
        "MonthlyIncome": np.round(rng.normal(5400, 2200, n_samples) * income_scale).clip(500, None),
        "NumberOfOpenCreditLinesAndLoans": rng.integers(0, 15, n_samples),
        "NumberOfTimes90DaysLate": rng.poisson(0.15 * delinquency_boost, n_samples),
        "NumberRealEstateLoansOrLines": rng.integers(0, 4, n_samples),
        "NumberOfTime60-89DaysPastDueNotWorse": rng.poisson(0.1 * delinquency_boost, n_samples),
        "NumberOfDependents": rng.integers(0, 5, n_samples),
    }

    # sprinkle a few missing values, like a real feed would have
    df = pd.DataFrame(data)
    miss_idx = rng.choice(n_samples, size=max(1, n_samples // 40), replace=False)
    df.loc[miss_idx, "MonthlyIncome"] = np.nan
    miss_idx2 = rng.choice(n_samples, size=max(1, n_samples // 60), replace=False)
    df.loc[miss_idx2, "NumberOfDependents"] = np.nan

    # --- schema drift injected at run 6: upstream renames a column and
    # adds a brand-new one ("CreditScoreBand") that our pipeline has never seen ---
    if run_number >= 6:
        df = df.rename(columns={"DebtRatio": "DebtRatioPct"})
        df["CreditScoreBand"] = rng.choice(["A", "B", "C", "D"], size=n_samples)

    df["batch_id"] = f"batch_{run_number:03d}"
    df["ingested_at"] = (BASE_DATE + timedelta(days=run_number)).isoformat()

    return df


## 4. Data drift metrics — PSI, KS-test, categorical drift

In [14]:
import numpy as np
import pandas as pd
from scipy import stats

PSI_WARN = 0.10
PSI_ALERT = 0.25
KS_ALERT_P = 0.01


def population_stability_index(reference: pd.Series, current: pd.Series, bins: int = 10) -> float:
    ref = reference.dropna().astype(float)
    cur = current.dropna().astype(float)
    if len(ref) < 5 or len(cur) < 5:
        return 0.0

    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(ref, quantiles))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf

    ref_counts, _ = np.histogram(ref, bins=edges)
    cur_counts, _ = np.histogram(cur, bins=edges)

    ref_pct = np.clip(ref_counts / max(ref_counts.sum(), 1), 1e-4, None)
    cur_pct = np.clip(cur_counts / max(cur_counts.sum(), 1), 1e-4, None)

    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return float(psi)


def ks_test(reference: pd.Series, current: pd.Series):
    ref = reference.dropna().astype(float)
    cur = current.dropna().astype(float)
    if len(ref) < 5 or len(cur) < 5:
        return 0.0, 1.0
    stat, p = stats.ks_2samp(ref, cur)
    return float(stat), float(p)


def categorical_drift(reference: pd.Series, current: pd.Series) -> float:
    """Total variation distance between category frequency distributions."""
    ref_counts = reference.value_counts(normalize=True)
    cur_counts = current.value_counts(normalize=True)
    all_cats = set(ref_counts.index) | set(cur_counts.index)
    tvd = 0.5 * sum(
        abs(ref_counts.get(c, 0.0) - cur_counts.get(c, 0.0)) for c in all_cats
    )
    return float(tvd)


def severity_from_psi(psi: float) -> str:
    if psi >= PSI_ALERT:
        return "alert"
    if psi >= PSI_WARN:
        return "warn"
    return "ok"


def compute_feature_drift(reference_df: pd.DataFrame, current_df: pd.DataFrame, numeric_cols, categorical_cols) -> list[dict]:
    """Returns a list of per-feature drift records."""
    records = []
    for col in numeric_cols:
        if col not in reference_df.columns or col not in current_df.columns:
            continue
        psi = population_stability_index(reference_df[col], current_df[col])
        ks_stat, ks_p = ks_test(reference_df[col], current_df[col])
        records.append({
            "feature": col,
            "type": "numeric",
            "psi": round(psi, 4),
            "ks_stat": round(ks_stat, 4),
            "ks_p_value": round(ks_p, 6),
            "severity": severity_from_psi(psi) if ks_p < KS_ALERT_P or psi >= PSI_WARN else "ok",
        })
    for col in categorical_cols:
        if col not in reference_df.columns or col not in current_df.columns:
            continue
        tvd = categorical_drift(reference_df[col], current_df[col])
        records.append({
            "feature": col,
            "type": "categorical",
            "psi": round(tvd, 4),  # store TVD in the same "psi" slot for uniform charting
            "ks_stat": None,
            "ks_p_value": None,
            "severity": severity_from_psi(tvd),
        })
    return records


## 5. Feature engineering (mirrors the source notebook)

In [15]:
import pandas as pd
import numpy as np


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # normalize the renamed debt-ratio column back to a canonical name so
    # downstream feature engineering keeps working even after the
    # upstream schema change (this is the kind of "pipeline resilience"
    # you'd add once schema drift is detected)
    debt_col = "DebtRatioPct" if "DebtRatioPct" in df.columns else "DebtRatio"

    # impute
    df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
    df["NumberOfDependents"] = df["NumberOfDependents"].fillna(df["NumberOfDependents"].median())

    # outlier treatment
    df["age"] = df["age"].apply(lambda x: 20 if x < 20 else x)

    # engineered features
    bins = [0, 24, 34, 44, 54, 64, max(df["age"].max() + 1, 65)]
    labels = ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
    df["Age_Band"] = pd.cut(df["age"], bins=bins, labels=labels, right=False)

    df["income_per_person"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)
    df["has_dependents"] = (df["NumberOfDependents"] > 0).astype(int)
    df["high_debt_ratio"] = (df[debt_col] > df[debt_col].median()).astype(int)
    df["total_past_due"] = (
        df["NumberOfTime30-59DaysPastDueNotWorse"]
        + df["NumberOfTimes90DaysLate"]
        + df["NumberOfTime60-89DaysPastDueNotWorse"]
    )
    return df


NUMERIC_FEATURES = [
    "RevolvingUtilizationOfUnsecuredLines", "age", "NumberOfTime30-59DaysPastDueNotWorse",
    "MonthlyIncome", "NumberOfOpenCreditLinesAndLoans", "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines", "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents", "income_per_person", "total_past_due",
]
CATEGORICAL_FEATURES = ["Age_Band", "has_dependents", "high_debt_ratio"]
TARGET = "SeriousDlqin2yrs"


## 6. SQLite-backed metadata & lineage store

In [16]:
import sqlite3
import json
from contextlib import contextmanager
from datetime import datetime

SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS runs (
    run_id INTEGER PRIMARY KEY,
    run_number INTEGER,
    started_at TEXT,
    finished_at TEXT,
    batch_id TEXT,
    batch_rows INTEGER,
    cumulative_rows INTEGER,
    status TEXT,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS lineage_edges (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id INTEGER,
    src_node TEXT,
    dst_node TEXT,
    src_kind TEXT,
    dst_kind TEXT,
    row_count INTEGER,
    checksum TEXT,
    FOREIGN KEY(run_id) REFERENCES runs(run_id)
);

CREATE TABLE IF NOT EXISTS schema_events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id INTEGER,
    added_columns TEXT,
    removed_columns TEXT,
    type_changed TEXT,
    has_drift INTEGER,
    FOREIGN KEY(run_id) REFERENCES runs(run_id)
);

CREATE TABLE IF NOT EXISTS drift_metrics (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id INTEGER,
    feature TEXT,
    feature_type TEXT,
    psi REAL,
    ks_stat REAL,
    ks_p_value REAL,
    severity TEXT,
    FOREIGN KEY(run_id) REFERENCES runs(run_id)
);

CREATE TABLE IF NOT EXISTS model_metrics (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id INTEGER,
    model_name TEXT,
    model_version TEXT,
    trained INTEGER,
    auc REAL,
    accuracy REAL,
    f1 REAL,
    n_train_rows INTEGER,
    FOREIGN KEY(run_id) REFERENCES runs(run_id)
);

CREATE TABLE IF NOT EXISTS fairness_metrics (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id INTEGER,
    group_name TEXT,
    metric_name TEXT,
    metric_value REAL,
    disparity REAL,
    FOREIGN KEY(run_id) REFERENCES runs(run_id)
);
"""


class MetadataStore:
    def __init__(self, path: str):
        self.path = path
        with self._conn() as conn:
            conn.executescript(SCHEMA_SQL)

    @contextmanager
    def _conn(self):
        conn = sqlite3.connect(self.path)
        try:
            yield conn
            conn.commit()
        finally:
            conn.close()

    def start_run(self, run_number: int, batch_id: str) -> int:
        with self._conn() as conn:
            cur = conn.execute(
                "INSERT INTO runs (run_number, started_at, batch_id, status) VALUES (?, ?, ?, ?)",
                (run_number, datetime.utcnow().isoformat(), batch_id, "running"),
            )
            return cur.lastrowid

    def finish_run(self, run_id: int, batch_rows: int, cumulative_rows: int, status: str = "success", notes: str = ""):
        with self._conn() as conn:
            conn.execute(
                "UPDATE runs SET finished_at=?, batch_rows=?, cumulative_rows=?, status=?, notes=? WHERE run_id=?",
                (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),
            )

    def log_lineage_edge(self, run_id: int, src_node, dst_node, src_kind, dst_kind, row_count=None, checksum=None):
        with self._conn() as conn:
            conn.execute(
                "INSERT INTO lineage_edges (run_id, src_node, dst_node, src_kind, dst_kind, row_count, checksum) "
                "VALUES (?, ?, ?, ?, ?, ?, ?)",
                (run_id, src_node, dst_node, src_kind, dst_kind, row_count, checksum),
            )

    def log_schema_event(self, run_id: int, diff):
        with self._conn() as conn:
            conn.execute(
                "INSERT INTO schema_events (run_id, added_columns, removed_columns, type_changed, has_drift) "
                "VALUES (?, ?, ?, ?, ?)",
                (
                    run_id,
                    json.dumps(diff.added_columns),
                    json.dumps(diff.removed_columns),
                    json.dumps(diff.type_changed),
                    int(diff.has_drift),
                ),
            )

    def log_drift_metrics(self, run_id: int, records: list[dict]):
        with self._conn() as conn:
            for r in records:
                conn.execute(
                    "INSERT INTO drift_metrics (run_id, feature, feature_type, psi, ks_stat, ks_p_value, severity) "
                    "VALUES (?, ?, ?, ?, ?, ?, ?)",
                    (run_id, r["feature"], r["type"], r["psi"], r["ks_stat"], r["ks_p_value"], r["severity"]),
                )

    def log_model_metrics(self, run_id: int, model_name, model_version, trained, auc, accuracy, f1, n_train_rows):
        with self._conn() as conn:
            conn.execute(
                "INSERT INTO model_metrics (run_id, model_name, model_version, trained, auc, accuracy, f1, n_train_rows) "
                "VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                (run_id, model_name, model_version, int(trained), auc, accuracy, f1, n_train_rows),
            )

    def log_fairness_metrics(self, run_id: int, records: list[dict]):
        with self._conn() as conn:
            for r in records:
                conn.execute(
                    "INSERT INTO fairness_metrics (run_id, group_name, metric_name, metric_value, disparity) "
                    "VALUES (?, ?, ?, ?, ?)",
                    (run_id, r["group_name"], r["metric_name"], r["metric_value"], r.get("disparity")),
                )

    # --- read helpers for the dashboard ---
    def fetch_all(self, query: str, params=()):
        import pandas as pd
        with self._conn() as conn:
            return pd.read_sql_query(query, conn, params=params)

store = MetadataStore(DB_PATH)
print('Metadata store ready at', DB_PATH)


Metadata store ready at /content/drive/MyDrive/credit_risk_pipeline/data/metadata.db


## 7. Pipeline orchestrator — one end-to-end run

In [17]:
RETRAIN_EVERY = 3

def _checksum(df: pd.DataFrame) -> str:
    return hashlib.md5(pd.util.hash_pandas_object(df, index=True).values).hexdigest()[:12]


def _build_preprocessor(numeric_cols, categorical_cols):
    return ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ])


def run_once(run_number: int, store: MetadataStore) -> dict:
    os.makedirs(BATCHES_DIR, exist_ok=True)

    # ---------- 1. ingest new batch ----------
    batch_df = generate_batch(run_number)
    batch_id = batch_df["batch_id"].iloc[0]
    batch_path = os.path.join(BATCHES_DIR, f"{batch_id}.csv")
    batch_df.to_csv(batch_path, index=False)

    run_id = store.start_run(run_number, batch_id)
    store.log_lineage_edge(run_id, "source_system", f"raw:{batch_id}", "external", "raw_batch",
                            row_count=len(batch_df), checksum=_checksum(batch_df))

    # ---------- 2. schema validation ----------
    observed_schema = snapshot_schema(batch_df)
    diff = diff_schema(EXPECTED_SCHEMA, observed_schema)
    store.log_schema_event(run_id, diff)

    # ---------- append to cumulative lake ----------
    lake_exists = os.path.exists(LAKE_PATH)
    if lake_exists:
        lake_df = pd.read_csv(LAKE_PATH)
        # union columns so schema-drifted batches don't crash the append
        combined = pd.concat([lake_df, batch_df], ignore_index=True, sort=False)
    else:
        combined = batch_df.copy()
    combined.to_csv(LAKE_PATH, index=False)

    store.log_lineage_edge(run_id, f"raw:{batch_id}", "lake:credit_risk_lake", "raw_batch", "data_lake",
                            row_count=len(combined), checksum=_checksum(combined))

    # ---------- 3. data drift vs reference (first batch ever seen) ----------
    reference_path = os.path.join(BATCHES_DIR, "batch_000.csv")
    numeric_for_drift = [c for c in ["RevolvingUtilizationOfUnsecuredLines", "age", "MonthlyIncome",
                                      "NumberOfOpenCreditLinesAndLoans", "NumberOfTimes90DaysLate"]
                         if c in batch_df.columns]
    categorical_for_drift = [c for c in ["CreditScoreBand"] if c in batch_df.columns]
    if os.path.exists(reference_path):
        ref_df = pd.read_csv(reference_path)
        drift_records = compute_feature_drift(ref_df, batch_df, numeric_for_drift, categorical_for_drift)
        store.log_drift_metrics(run_id, drift_records)

    # ---------- 4. feature engineering on the FULL cumulative lake ----------
    fe_df = engineer_features(combined)
    store.log_lineage_edge(run_id, "lake:credit_risk_lake", "features:engineered", "data_lake", "feature_set",
                            row_count=len(fe_df))

    X = fe_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y = fe_df[TARGET]

    # ---------- 5. train (periodically) or reuse existing model ----------
    trained_this_run = (run_number % RETRAIN_EVERY == 0) or (not os.path.exists(MODEL_PATH))
    if trained_this_run:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
        preprocessor = _build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
        X_train_proc = preprocessor.fit_transform(X_train)
        X_test_proc = preprocessor.transform(X_test)

        model = LogisticRegression(solver="liblinear", class_weight="balanced", random_state=42)
        model.fit(X_train_proc, y_train)

        y_pred = model.predict(X_test_proc)
        y_proba = model.predict_proba(X_test_proc)[:, 1]
        auc = roc_auc_score(y_test, y_proba) if len(set(y_test)) > 1 else float("nan")
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        joblib.dump({"model": model, "preprocessor": preprocessor,
                     "numeric": NUMERIC_FEATURES, "categorical": CATEGORICAL_FEATURES},
                    MODEL_PATH)

        model_version = f"v{run_number}"
        store.log_model_metrics(run_id, "LogisticRegression", model_version, True, auc, acc, f1, len(X_train))
        store.log_lineage_edge(run_id, "features:engineered", f"model:{model_version}", "feature_set", "model",
                                row_count=len(X_train))

        # ---------- 6. fairness assessment (on this run's held-out test set) ----------
        age_band_test = X_test["Age_Band"].astype(str)
        sensitive = pd.DataFrame({
            "is_65_plus": (age_band_test == "65+").astype(int),
            "is_under_25": (age_band_test == "<25").astype(int),
        })
        fairness_records = []
        for col in sensitive.columns:
            mf = MetricFrame(
                metrics={"accuracy": accuracy_score},
                y_true=y_test, y_pred=y_pred, sensitive_features=sensitive[col],
            )
            try:
                dpd = demographic_parity_difference(y_test, y_pred, sensitive_features=sensitive[col])
            except Exception:
                dpd = float("nan")
            for group_val, acc_val in mf.by_group["accuracy"].items():
                fairness_records.append({
                    "group_name": f"{col}={group_val}",
                    "metric_name": "accuracy",
                    "metric_value": float(acc_val),
                    "disparity": float(dpd),
                })
        store.log_fairness_metrics(run_id, fairness_records)
    else:
        store.log_model_metrics(run_id, "LogisticRegression", "reused", False, None, None, None, len(X))
        store.log_lineage_edge(run_id, "features:engineered", "model:reused", "feature_set", "model", row_count=len(X))

    store.log_lineage_edge(run_id, f"model:{'v'+str(run_number) if trained_this_run else 'reused'}",
                            f"predictions:{batch_id}", "model", "predictions", row_count=len(batch_df))

    store.finish_run(run_id, batch_rows=len(batch_df), cumulative_rows=len(combined), status="success")
    return {"run_id": run_id, "run_number": run_number, "batch_id": batch_id,
            "batch_rows": len(batch_df), "cumulative_rows": len(combined),
            "schema_drift": diff.has_drift, "trained": trained_this_run}


## 8. Run the simulation

Set `reset=True` **only** the first time (wipes Drive data + metadata and
starts from run 0). Every subsequent time, run with `reset=False` -- it
continues from the last run number in the metadata store, appending new
batches, just like a real recurring job.

In [18]:
def run_simulation(n_runs: int, reset: bool = False):
    global store
    if reset:
        import shutil
        for p in [DB_PATH, LAKE_PATH, MODEL_PATH]:
            if os.path.exists(p):
                os.remove(p)
        if os.path.exists(BATCHES_DIR):
            shutil.rmtree(BATCHES_DIR)
        os.makedirs(BATCHES_DIR, exist_ok=True)
        store = MetadataStore(DB_PATH)

    existing = store.fetch_all("SELECT MAX(run_number) as last_run FROM runs")
    last_run = existing["last_run"].iloc[0]
    start_run = 0 if last_run is None or last_run != last_run else int(last_run) + 1

    print(f"Continuing from run #{start_run} (simulating {n_runs} new incoming batches)\n")
    header = f"{'run':>4}  {'batch':>10}  {'batch_rows':>10}  {'cumulative':>10}  {'schema_drift':>12}  {'trained':>7}"
    print(header)
    for run_number in range(start_run, start_run + n_runs):
        result = run_once(run_number, store)
        line = (f"{result['run_number']:>4}  {result['batch_id']:>10}  {result['batch_rows']:>10}  "
                f"{result['cumulative_rows']:>10}  {str(result['schema_drift']):>12}  {str(result['trained']):>7}")
        print(line)
    print(f"\nDone. Metadata store at {DB_PATH}")

# First run: reset=True, simulate 10 days of batches
run_simulation(n_runs=10, reset=True)


Continuing from run #0 (simulating 10 new incoming batches)

 run       batch  batch_rows  cumulative  schema_drift  trained


/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),
/tmp/ipykernel_2355/4065970408.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),
/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),
/tmp/ipykernel_2355/4065970408.py:105: DeprecationWar

   0   batch_000         250         250         False     True
   1   batch_001         250         500         False    False


/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),
/tmp/ipykernel_2355/4065970408.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),
/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),


   2   batch_002         250         750         False    False


...

/tmp/ipykernel_2355/4065970408.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),
/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),


   7   batch_007         250        2000          True    False


/tmp/ipykernel_2355/4065970408.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),
/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),


   8   batch_008         250        2250          True    False
   9   batch_009         250        2500          True     True

Done. Metadata store at /content/drive/MyDrive/credit_risk_pipeline/data/metadata.db


/tmp/ipykernel_2355/4065970408.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),


In [19]:
# Later, to simulate the next few days arriving, just re-run this
# (leave reset=False so it continues rather than wiping history):
run_simulation(n_runs=4, reset=False)

/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),


Continuing from run #10 (simulating 4 new incoming batches)

 run       batch  batch_rows  cumulative  schema_drift  trained


/tmp/ipykernel_2355/4065970408.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),
/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),


  10   batch_010         250        2750          True    False


/tmp/ipykernel_2355/4065970408.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),
/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),


  11   batch_011         250        3000          True    False


/tmp/ipykernel_2355/4065970408.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),
/tmp/ipykernel_2355/4065970408.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (run_number, datetime.utcnow().isoformat(), batch_id, "running"),


  12   batch_012         250        3250          True     True
  13   batch_013         250        3500          True    False

Done. Metadata store at /content/drive/MyDrive/credit_risk_pipeline/data/metadata.db


/tmp/ipykernel_2355/4065970408.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (datetime.utcnow().isoformat(), batch_rows, cumulative_rows, status, notes, run_id),


## 9. Build & display the observability dashboard

In [20]:
def render_html(ctx: dict) -> str:
    data_json = json.dumps(ctx)
    n_schema_alerts = len(ctx["schema_alerts"])
    n_drift_alerts = sum(1 for x in ctx["latest_severity"] if x["severity"] == "alert")
    n_drift_warns = sum(1 for x in ctx["latest_severity"] if x["severity"] == "warn")

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Pipeline Observability Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.4/chart.umd.min.js"></script>
<style>
  :root {{
    --bg: #0b1020; --panel: #131a2e; --panel-2: #1a2340; --border: #26314f;
    --text: #e8ecf8; --muted: #92a0c4; --accent: #6ea8fe; --ok: #35d399;
    --warn: #f5b942; --alert: #ff5c72;
  }}
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0; background: var(--bg); color: var(--text);
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
  }}
  header {{
    padding: 24px 32px; border-bottom: 1px solid var(--border);
    display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 12px;
  }}
  header h1 {{ margin: 0; font-size: 20px; font-weight: 600; }}
  header p {{ margin: 4px 0 0; color: var(--muted); font-size: 13px; }}
  .stat-row {{ display: flex; gap: 12px; flex-wrap: wrap; }}
  .stat {{
    background: var(--panel); border: 1px solid var(--border); border-radius: 10px;
    padding: 10px 16px; min-width: 110px; text-align: center;
  }}
  .stat .num {{ font-size: 22px; font-weight: 700; }}
  .stat .label {{ font-size: 11px; color: var(--muted); text-transform: uppercase; letter-spacing: .04em; }}
  .stat.alert .num {{ color: var(--alert); }}
  .stat.warn .num {{ color: var(--warn); }}
  .stat.ok .num {{ color: var(--ok); }}
  main {{ padding: 24px 32px; display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
  .full {{ grid-column: 1 / -1; }}
  .card {{
    background: var(--panel); border: 1px solid var(--border); border-radius: 14px; padding: 18px 20px;
  }}
  .card h2 {{ margin: 0 0 4px; font-size: 14px; font-weight: 600; }}
  .card .sub {{ margin: 0 0 14px; font-size: 12px; color: var(--muted); }}
  table {{ width: 100%; border-collapse: collapse; font-size: 12.5px; }}
  th, td {{ text-align: left; padding: 6px 8px; border-bottom: 1px solid var(--border); }}
  th {{ color: var(--muted); font-weight: 500; text-transform: uppercase; font-size: 10.5px; letter-spacing: .03em; }}
  tr:hover td {{ background: rgba(255,255,255,0.02); }}
  .badge {{ padding: 2px 8px; border-radius: 100px; font-size: 11px; font-weight: 600; display: inline-block; }}
  .badge.ok {{ background: rgba(53,211,153,0.15); color: var(--ok); }}
  .badge.warn {{ background: rgba(245,185,66,0.15); color: var(--warn); }}
  .badge.alert {{ background: rgba(255,92,114,0.15); color: var(--alert); }}
  .alert-box {{
    border: 1px solid rgba(255,92,114,0.4); background: rgba(255,92,114,0.08);
    border-radius: 10px; padding: 10px 14px; margin-bottom: 10px; font-size: 13px;
  }}
  .alert-box b {{ color: var(--alert); }}
  .no-alerts {{ color: var(--muted); font-size: 13px; }}
  .lineage {{ display: flex; align-items: center; flex-wrap: wrap; gap: 6px; font-size: 12.5px; }}
  .node {{
    background: var(--panel-2); border: 1px solid var(--border); border-radius: 8px;
    padding: 6px 10px; white-space: nowrap;
  }}
  .arrow {{ color: var(--muted); }}
  canvas {{ max-height: 260px; }}
  .scroll {{ max-height: 260px; overflow-y: auto; }}
</style>
</head>
<body>
<header>
  <div>
    <h1>🔎 Credit Risk Pipeline — Observability Dashboard</h1>
    <p>Simulated batch pipeline · {ctx['total_runs']} runs · {ctx['total_rows']:,} cumulative rows ingested</p>
  </div>
  <div class="stat-row">
    <div class="stat {'alert' if n_schema_alerts else 'ok'}">
      <div class="num">{n_schema_alerts}</div><div class="label">Schema Changes</div>
    </div>
    <div class="stat {'alert' if n_drift_alerts else ('warn' if n_drift_warns else 'ok')}">
      <div class="num">{n_drift_alerts}</div><div class="label">Features Drift-Alert</div>
    </div>
    <div class="stat warn">
      <div class="num">{n_drift_warns}</div><div class="label">Features Drift-Warn</div>
    </div>
    <div class="stat ok">
      <div class="num">{ctx['latest_run']}</div><div class="label">Latest Run #</div>
    </div>
  </div>
</header>

<main>
  <div class="card full">
    <h2>Schema Drift Alerts</h2>
    <p class="sub">Detected changes vs. the expected raw-ingestion schema, first run each change was observed</p>
    <div id="schema-alerts"></div>
  </div>

  <div class="card">
    <h2>Data Volume Growth</h2>
    <p class="sub">Per-batch and cumulative row counts in the data lake</p>
    <canvas id="volumeChart"></canvas>
  </div>

  <div class="card">
    <h2>Feature Drift (PSI) Over Time</h2>
    <p class="sub">Population Stability Index per feature · &gt;0.10 warn, &gt;0.25 alert</p>
    <canvas id="driftChart"></canvas>
  </div>

  <div class="card">
    <h2>Model Performance Trend</h2>
    <p class="sub">Metrics recorded on retrain runs only (held-out test split)</p>
    <canvas id="modelChart"></canvas>
  </div>

  <div class="card">
    <h2>Fairness — Demographic Parity Disparity</h2>
    <p class="sub">Gap in positive-prediction rate across sensitive groups (lower is more equitable)</p>
    <canvas id="fairnessChart"></canvas>
  </div>

  <div class="card full">
    <h2>Lineage — Latest Run (#{ctx['latest_run']})</h2>
    <p class="sub">Source → raw batch → data lake → feature set → model → predictions</p>
    <div id="lineage" class="lineage"></div>
  </div>

  <div class="card full">
    <h2>Run Log</h2>
    <p class="sub">Full history of pipeline executions</p>
    <div class="scroll">
      <table id="runs-table">
        <thead><tr><th>Run</th><th>Batch</th><th>Batch Rows</th><th>Cumulative</th><th>Status</th><th>Finished</th></tr></thead>
        <tbody></tbody>
      </table>
    </div>
  </div>
</main>

<script>
const DATA = {data_json};
const colors = ['#6ea8fe','#35d399','#f5b942','#ff5c72','#c792ea','#5fd0d0','#f78fb3','#a3e635'];

// --- schema alerts ---
const schemaDiv = document.getElementById('schema-alerts');
if (DATA.schema_alerts.length === 0) {{
  schemaDiv.innerHTML = '<p class="no-alerts">No schema drift detected across any run.</p>';
}} else {{
  schemaDiv.innerHTML = DATA.schema_alerts.map(a => {{
    let parts = [];
    if (a.added.length) parts.push(`<b>+ added:</b> ${{a.added.join(', ')}}`);
    if (a.removed.length) parts.push(`<b>− removed:</b> ${{a.removed.join(', ')}}`);
    if (a.type_changed.length) parts.push(`<b>type changed:</b> ${{a.type_changed.map(t => t.join(' ')).join(', ')}}`);
    return `<div class="alert-box"><b>Run #${{a.run_number}}</b> — ${{parts.join(' &nbsp;|&nbsp; ')}}</div>`;
  }}).join('');
}}

// --- lineage ---
const lineageDiv = document.getElementById('lineage');
lineageDiv.innerHTML = DATA.latest_lineage.map((e, i) => {{
  const arrow = i < DATA.latest_lineage.length ? '<span class="arrow">→</span>' : '';
  return `<span class="node">${{e.src_node}}</span>${{arrow}}<span class="node">${{e.dst_node}}</span>${{i < DATA.latest_lineage.length - 1 ? '<span class="arrow">→</span>' : ''}}`;
}}).join('');

// --- runs table ---
const tbody = document.querySelector('#runs-table tbody');
tbody.innerHTML = DATA.runs_table.slice().reverse().map(r => `
  <tr>
    <td>${{r.run_number}}</td>
    <td>${{r.batch_id}}</td>
    <td>${{r.batch_rows}}</td>
    <td>${{r.cumulative_rows}}</td>
    <td><span class="badge ok">${{r.status}}</span></td>
    <td>${{(r.finished_at || '').replace('T',' ').slice(0,19)}}</td>
  </tr>`).join('');

const chartDefaults = {{
  responsive: true,
  maintainAspectRatio: false,
  plugins: {{ legend: {{ labels: {{ color: '#e8ecf8', font: {{ size: 11 }} }} }} }},
  scales: {{
    x: {{ ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }} }},
    y: {{ ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }} }}
  }}
}};

// --- volume chart ---
new Chart(document.getElementById('volumeChart'), {{
  type: 'bar',
  data: {{
    labels: DATA.volume.runs,
    datasets: [
      {{ label: 'Batch rows', data: DATA.volume.batch_rows, backgroundColor: '#6ea8fe', order: 2 }},
      {{ label: 'Cumulative rows', data: DATA.volume.cumulative_rows, type: 'line', borderColor: '#35d399',
         backgroundColor: 'transparent', yAxisID: 'y1', order: 1, tension: 0.3 }}
    ]
  }},
  options: {{ ...chartDefaults,
    scales: {{ ...chartDefaults.scales,
      y1: {{ position: 'right', ticks: {{ color: '#92a0c4' }}, grid: {{ display: false }} }}
    }}
  }}
}});

// --- drift chart ---
const driftDatasets = Object.keys(DATA.drift_series).map((feat, i) => ({{
  label: feat,
  data: DATA.drift_series[feat].runs.map((r, idx) => ({{ x: r, y: DATA.drift_series[feat].psi[idx] }})),
  borderColor: colors[i % colors.length],
  backgroundColor: 'transparent',
  tension: 0.3,
}}));
new Chart(document.getElementById('driftChart'), {{
  type: 'line',
  data: {{ datasets: driftDatasets }},
  options: {{ ...chartDefaults,
    scales: {{ x: {{ type: 'linear', ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }}, title: {{ display: true, text: 'run #', color: '#92a0c4' }} }},
               y: {{ ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }}, title: {{ display: true, text: 'PSI', color: '#92a0c4' }} }} }}
  }}
}});

// --- model chart ---
new Chart(document.getElementById('modelChart'), {{
  type: 'line',
  data: {{
    labels: DATA.model_trend.runs,
    datasets: [
      {{ label: 'AUC', data: DATA.model_trend.auc, borderColor: '#6ea8fe', backgroundColor: 'transparent', tension: 0.3 }},
      {{ label: 'Accuracy', data: DATA.model_trend.accuracy, borderColor: '#35d399', backgroundColor: 'transparent', tension: 0.3 }},
      {{ label: 'F1', data: DATA.model_trend.f1, borderColor: '#f5b942', backgroundColor: 'transparent', tension: 0.3 }}
    ]
  }},
  options: chartDefaults
}});

// --- fairness chart ---
const fairnessDatasets = Object.keys(DATA.fairness_trend).map((attr, i) => ({{
  label: attr,
  data: DATA.fairness_trend[attr].runs.map((r, idx) => ({{ x: r, y: DATA.fairness_trend[attr].disparity[idx] }})),
  borderColor: colors[(i + 3) % colors.length],
  backgroundColor: 'transparent',
  tension: 0.3,
}}));
new Chart(document.getElementById('fairnessChart'), {{
  type: 'line',
  data: {{ datasets: fairnessDatasets }},
  options: {{ ...chartDefaults,
    scales: {{ x: {{ type: 'linear', ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }}, title: {{ display: true, text: 'run #', color: '#92a0c4' }} }},
               y: {{ ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }}, title: {{ display: true, text: 'disparity', color: '#92a0c4' }} }} }}
  }}
}});
</script>
</body>
</html>
"""




def build_dashboard():

    runs = store.fetch_all("SELECT * FROM runs ORDER BY run_number")
    schema_events = store.fetch_all("SELECT * FROM schema_events ORDER BY run_id")
    drift = store.fetch_all("""
        SELECT r.run_number, d.feature, d.feature_type, d.psi, d.ks_p_value, d.severity
        FROM drift_metrics d JOIN runs r ON d.run_id = r.run_id
        ORDER BY r.run_number
    """)
    model_metrics = store.fetch_all("""
        SELECT r.run_number, m.model_version, m.trained, m.auc, m.accuracy, m.f1, m.n_train_rows
        FROM model_metrics m JOIN runs r ON m.run_id = r.run_id
        ORDER BY r.run_number
    """)
    fairness = store.fetch_all("""
        SELECT r.run_number, f.group_name, f.metric_name, f.metric_value, f.disparity
        FROM fairness_metrics f JOIN runs r ON f.run_id = r.run_id
        ORDER BY r.run_number
    """)
    lineage = store.fetch_all("""
        SELECT r.run_number, l.src_node, l.dst_node, l.src_kind, l.dst_kind, l.row_count
        FROM lineage_edges l JOIN runs r ON l.run_id = r.run_id
        ORDER BY r.run_number, l.id
    """)

    latest_run = int(runs["run_number"].max())

    # --- schema drift alerts (first run each new drift signature appears) ---
    schema_alert_runs = schema_events[schema_events["has_drift"] == 1].copy()
    schema_alert_runs = schema_alert_runs.merge(runs[["run_id", "run_number"]], on="run_id")
    schema_alerts = []
    seen_signature = None
    for _, row in schema_alert_runs.sort_values("run_number").iterrows():
        sig = (row["added_columns"], row["removed_columns"], row["type_changed"])
        if sig != seen_signature:
            schema_alerts.append({
                "run_number": int(row["run_number"]),
                "added": json.loads(row["added_columns"]),
                "removed": json.loads(row["removed_columns"]),
                "type_changed": json.loads(row["type_changed"]),
            })
            seen_signature = sig

    # --- feature drift trend series, one line per feature ---
    drift_features = sorted(drift["feature"].unique().tolist())
    drift_series = {
        feat: {
            "runs": drift[drift["feature"] == feat]["run_number"].tolist(),
            "psi": drift[drift["feature"] == feat]["psi"].tolist(),
        }
        for feat in drift_features
    }
    latest_severity = (
        drift.sort_values("run_number").groupby("feature").tail(1)[["feature", "severity", "psi"]]
        .to_dict("records")
    )

    # --- model metric trend (only rows where a training actually happened) ---
    trained_rows = model_metrics[model_metrics["trained"] == 1]
    model_trend = {
        "runs": trained_rows["run_number"].tolist(),
        "auc": trained_rows["auc"].round(4).tolist(),
        "accuracy": trained_rows["accuracy"].round(4).tolist(),
        "f1": trained_rows["f1"].round(4).tolist(),
    }

    # --- fairness trend: demographic parity disparity per sensitive attribute over time ---
    fairness_disparity = fairness.drop_duplicates(subset=["run_number", "group_name"])
    fairness_disparity["attr"] = fairness_disparity["group_name"].str.split("=").str[0]
    fairness_trend = {}
    for attr in fairness_disparity["attr"].unique():
        sub = fairness_disparity[fairness_disparity["attr"] == attr].drop_duplicates(subset="run_number")
        fairness_trend[attr] = {
            "runs": sub["run_number"].tolist(),
            "disparity": sub["disparity"].round(4).tolist(),
        }

    # --- lineage graph for the latest run ---
    latest_lineage = lineage[lineage["run_number"] == latest_run].to_dict("records")

    # --- volume growth ---
    volume = {
        "runs": runs["run_number"].tolist(),
        "batch_rows": runs["batch_rows"].tolist(),
        "cumulative_rows": runs["cumulative_rows"].tolist(),
    }

    context = {
        "latest_run": latest_run,
        "total_runs": len(runs),
        "total_rows": int(runs["cumulative_rows"].max()),
        "schema_alerts": schema_alerts,
        "drift_series": drift_series,
        "latest_severity": latest_severity,
        "model_trend": model_trend,
        "fairness_trend": fairness_trend,
        "latest_lineage": latest_lineage,
        "volume": volume,
        "runs_table": runs.fillna("").to_dict("records"),
    }

    html = render_html(context)
    with open(DASHBOARD_PATH, "w") as f:
        f.write(html)
    print("Dashboard written to", DASHBOARD_PATH)
    return html


def render_html(ctx: dict) -> str:
    data_json = json.dumps(ctx)
    n_schema_alerts = len(ctx["schema_alerts"])
    n_drift_alerts = sum(1 for x in ctx["latest_severity"] if x["severity"] == "alert")
    n_drift_warns = sum(1 for x in ctx["latest_severity"] if x["severity"] == "warn")

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Pipeline Observability Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.4/chart.umd.min.js"></script>
<style>
  :root {{
    --bg: #0b1020; --panel: #131a2e; --panel-2: #1a2340; --border: #26314f;
    --text: #e8ecf8; --muted: #92a0c4; --accent: #6ea8fe; --ok: #35d399;
    --warn: #f5b942; --alert: #ff5c72;
  }}
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0; background: var(--bg); color: var(--text);
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
  }}
  header {{
    padding: 24px 32px; border-bottom: 1px solid var(--border);
    display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 12px;
  }}
  header h1 {{ margin: 0; font-size: 20px; font-weight: 600; }}
  header p {{ margin: 4px 0 0; color: var(--muted); font-size: 13px; }}
  .stat-row {{ display: flex; gap: 12px; flex-wrap: wrap; }}
  .stat {{
    background: var(--panel); border: 1px solid var(--border); border-radius: 10px;
    padding: 10px 16px; min-width: 110px; text-align: center;
  }}
  .stat .num {{ font-size: 22px; font-weight: 700; }}
  .stat .label {{ font-size: 11px; color: var(--muted); text-transform: uppercase; letter-spacing: .04em; }}
  .stat.alert .num {{ color: var(--alert); }}
  .stat.warn .num {{ color: var(--warn); }}
  .stat.ok .num {{ color: var(--ok); }}
  main {{ padding: 24px 32px; display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
  .full {{ grid-column: 1 / -1; }}
  .card {{
    background: var(--panel); border: 1px solid var(--border); border-radius: 14px; padding: 18px 20px;
  }}
  .card h2 {{ margin: 0 0 4px; font-size: 14px; font-weight: 600; }}
  .card .sub {{ margin: 0 0 14px; font-size: 12px; color: var(--muted); }}
  table {{ width: 100%; border-collapse: collapse; font-size: 12.5px; }}
  th, td {{ text-align: left; padding: 6px 8px; border-bottom: 1px solid var(--border); }}
  th {{ color: var(--muted); font-weight: 500; text-transform: uppercase; font-size: 10.5px; letter-spacing: .03em; }}
  tr:hover td {{ background: rgba(255,255,255,0.02); }}
  .badge {{ padding: 2px 8px; border-radius: 100px; font-size: 11px; font-weight: 600; display: inline-block; }}
  .badge.ok {{ background: rgba(53,211,153,0.15); color: var(--ok); }}
  .badge.warn {{ background: rgba(245,185,66,0.15); color: var(--warn); }}
  .badge.alert {{ background: rgba(255,92,114,0.15); color: var(--alert); }}
  .alert-box {{
    border: 1px solid rgba(255,92,114,0.4); background: rgba(255,92,114,0.08);
    border-radius: 10px; padding: 10px 14px; margin-bottom: 10px; font-size: 13px;
  }}
  .alert-box b {{ color: var(--alert); }}
  .no-alerts {{ color: var(--muted); font-size: 13px; }}
  .lineage {{ display: flex; align-items: center; flex-wrap: wrap; gap: 6px; font-size: 12.5px; }}
  .node {{
    background: var(--panel-2); border: 1px solid var(--border); border-radius: 8px;
    padding: 6px 10px; white-space: nowrap;
  }}
  .arrow {{ color: var(--muted); }}
  canvas {{ max-height: 260px; }}
  .scroll {{ max-height: 260px; overflow-y: auto; }}
</style>
</head>
<body>
<header>
  <div>
    <h1>🔎 Credit Risk Pipeline — Observability Dashboard</h1>
    <p>Simulated batch pipeline · {ctx['total_runs']} runs · {ctx['total_rows']:,} cumulative rows ingested</p>
  </div>
  <div class="stat-row">
    <div class="stat {'alert' if n_schema_alerts else 'ok'}">
      <div class="num">{n_schema_alerts}</div><div class="label">Schema Changes</div>
    </div>
    <div class="stat {'alert' if n_drift_alerts else ('warn' if n_drift_warns else 'ok')}">
      <div class="num">{n_drift_alerts}</div><div class="label">Features Drift-Alert</div>
    </div>
    <div class="stat warn">
      <div class="num">{n_drift_warns}</div><div class="label">Features Drift-Warn</div>
    </div>
    <div class="stat ok">
      <div class="num">{ctx['latest_run']}</div><div class="label">Latest Run #</div>
    </div>
  </div>
</header>

<main>
  <div class="card full">
    <h2>Schema Drift Alerts</h2>
    <p class="sub">Detected changes vs. the expected raw-ingestion schema, first run each change was observed</p>
    <div id="schema-alerts"></div>
  </div>

  <div class="card">
    <h2>Data Volume Growth</h2>
    <p class="sub">Per-batch and cumulative row counts in the data lake</p>
    <canvas id="volumeChart"></canvas>
  </div>

  <div class="card">
    <h2>Feature Drift (PSI) Over Time</h2>
    <p class="sub">Population Stability Index per feature · &gt;0.10 warn, &gt;0.25 alert</p>
    <canvas id="driftChart"></canvas>
  </div>

  <div class="card">
    <h2>Model Performance Trend</h2>
    <p class="sub">Metrics recorded on retrain runs only (held-out test split)</p>
    <canvas id="modelChart"></canvas>
  </div>

  <div class="card">
    <h2>Fairness — Demographic Parity Disparity</h2>
    <p class="sub">Gap in positive-prediction rate across sensitive groups (lower is more equitable)</p>
    <canvas id="fairnessChart"></canvas>
  </div>

  <div class="card full">
    <h2>Lineage — Latest Run (#{ctx['latest_run']})</h2>
    <p class="sub">Source → raw batch → data lake → feature set → model → predictions</p>
    <div id="lineage" class="lineage"></div>
  </div>

  <div class="card full">
    <h2>Run Log</h2>
    <p class="sub">Full history of pipeline executions</p>
    <div class="scroll">
      <table id="runs-table">
        <thead><tr><th>Run</th><th>Batch</th><th>Batch Rows</th><th>Cumulative</th><th>Status</th><th>Finished</th></tr></thead>
        <tbody></tbody>
      </table>
    </div>
  </div>
</main>

<script>
const DATA = {data_json};
const colors = ['#6ea8fe','#35d399','#f5b942','#ff5c72','#c792ea','#5fd0d0','#f78fb3','#a3e635'];

// --- schema alerts ---
const schemaDiv = document.getElementById('schema-alerts');
if (DATA.schema_alerts.length === 0) {{
  schemaDiv.innerHTML = '<p class="no-alerts">No schema drift detected across any run.</p>';
}} else {{
  schemaDiv.innerHTML = DATA.schema_alerts.map(a => {{
    let parts = [];
    if (a.added.length) parts.push(`<b>+ added:</b> ${{a.added.join(', ')}}`);
    if (a.removed.length) parts.push(`<b>− removed:</b> ${{a.removed.join(', ')}}`);
    if (a.type_changed.length) parts.push(`<b>type changed:</b> ${{a.type_changed.map(t => t.join(' ')).join(', ')}}`);
    return `<div class="alert-box"><b>Run #${{a.run_number}}</b> — ${{parts.join(' &nbsp;|&nbsp; ')}}</div>`;
  }}).join('');
}}

// --- lineage ---
const lineageDiv = document.getElementById('lineage');
lineageDiv.innerHTML = DATA.latest_lineage.map((e, i) => {{
  const arrow = i < DATA.latest_lineage.length ? '<span class="arrow">→</span>' : '';
  return `<span class="node">${{e.src_node}}</span>${{arrow}}<span class="node">${{e.dst_node}}</span>${{i < DATA.latest_lineage.length - 1 ? '<span class="arrow">→</span>' : ''}}`;
}}).join('');

// --- runs table ---
const tbody = document.querySelector('#runs-table tbody');
tbody.innerHTML = DATA.runs_table.slice().reverse().map(r => `
  <tr>
    <td>${{r.run_number}}</td>
    <td>${{r.batch_id}}</td>
    <td>${{r.batch_rows}}</td>
    <td>${{r.cumulative_rows}}</td>
    <td><span class="badge ok">${{r.status}}</span></td>
    <td>${{(r.finished_at || '').replace('T',' ').slice(0,19)}}</td>
  </tr>`).join('');

const chartDefaults = {{
  responsive: true,
  maintainAspectRatio: false,
  plugins: {{ legend: {{ labels: {{ color: '#e8ecf8', font: {{ size: 11 }} }} }} }},
  scales: {{
    x: {{ ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }} }},
    y: {{ ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }} }}
  }}
}};

// --- volume chart ---
new Chart(document.getElementById('volumeChart'), {{
  type: 'bar',
  data: {{
    labels: DATA.volume.runs,
    datasets: [
      {{ label: 'Batch rows', data: DATA.volume.batch_rows, backgroundColor: '#6ea8fe', order: 2 }},
      {{ label: 'Cumulative rows', data: DATA.volume.cumulative_rows, type: 'line', borderColor: '#35d399',
         backgroundColor: 'transparent', yAxisID: 'y1', order: 1, tension: 0.3 }}
    ]
  }},
  options: {{ ...chartDefaults,
    scales: {{ ...chartDefaults.scales,
      y1: {{ position: 'right', ticks: {{ color: '#92a0c4' }}, grid: {{ display: false }} }}
    }}
  }}
}});

// --- drift chart ---
const driftDatasets = Object.keys(DATA.drift_series).map((feat, i) => ({{
  label: feat,
  data: DATA.drift_series[feat].runs.map((r, idx) => ({{ x: r, y: DATA.drift_series[feat].psi[idx] }})),
  borderColor: colors[i % colors.length],
  backgroundColor: 'transparent',
  tension: 0.3,
}}));
new Chart(document.getElementById('driftChart'), {{
  type: 'line',
  data: {{ datasets: driftDatasets }},
  options: {{ ...chartDefaults,
    scales: {{ x: {{ type: 'linear', ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }}, title: {{ display: true, text: 'run #', color: '#92a0c4' }} }},
               y: {{ ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }}, title: {{ display: true, text: 'PSI', color: '#92a0c4' }} }} }}
  }}
}});

// --- model chart ---
new Chart(document.getElementById('modelChart'), {{
  type: 'line',
  data: {{
    labels: DATA.model_trend.runs,
    datasets: [
      {{ label: 'AUC', data: DATA.model_trend.auc, borderColor: '#6ea8fe', backgroundColor: 'transparent', tension: 0.3 }},
      {{ label: 'Accuracy', data: DATA.model_trend.accuracy, borderColor: '#35d399', backgroundColor: 'transparent', tension: 0.3 }},
      {{ label: 'F1', data: DATA.model_trend.f1, borderColor: '#f5b942', backgroundColor: 'transparent', tension: 0.3 }}
    ]
  }},
  options: chartDefaults
}});

// --- fairness chart ---
const fairnessDatasets = Object.keys(DATA.fairness_trend).map((attr, i) => ({{
  label: attr,
  data: DATA.fairness_trend[attr].runs.map((r, idx) => ({{ x: r, y: DATA.fairness_trend[attr].disparity[idx] }})),
  borderColor: colors[(i + 3) % colors.length],
  backgroundColor: 'transparent',
  tension: 0.3,
}}));
new Chart(document.getElementById('fairnessChart'), {{
  type: 'line',
  data: {{ datasets: fairnessDatasets }},
  options: {{ ...chartDefaults,
    scales: {{ x: {{ type: 'linear', ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }}, title: {{ display: true, text: 'run #', color: '#92a0c4' }} }},
               y: {{ ticks: {{ color: '#92a0c4' }}, grid: {{ color: '#26314f' }}, title: {{ display: true, text: 'disparity', color: '#92a0c4' }} }} }}
  }}
}});
</script>
</body>
</html>
"""




dashboard_html = build_dashboard()
from IPython.display import HTML, display
display(HTML(dashboard_html))


Dashboard written to /content/drive/MyDrive/credit_risk_pipeline/dashboard.html


Run,Batch,Batch Rows,Cumulative,Status,Finished


## 10. Query the metadata store directly (optional)

In [21]:
display(store.fetch_all("SELECT run_number, batch_id, batch_rows, cumulative_rows, status FROM runs ORDER BY run_number"))

,run_number,batch_id,batch_rows,cumulative_rows,status
0,0,batch_000,250,250,success
1,1,batch_001,250,500,success
2,2,batch_002,250,750,success
3,3,batch_003,250,1000,success
4,4,batch_004,250,1250,success
5,5,batch_005,250,1500,success
6,6,batch_006,250,1750,success
7,7,batch_007,250,2000,success
8,8,batch_008,250,2250,success
9,9,batch_009,250,2500,success


In [22]:
display(store.fetch_all('''
    SELECT r.run_number, d.feature, d.psi, d.severity
    FROM drift_metrics d JOIN runs r ON d.run_id = r.run_id
    WHERE d.severity != 'ok'
    ORDER BY r.run_number
'''))

,run_number,feature,psi,severity
0,2,RevolvingUtilizationOfUnsecuredLines,0.1614,warn
1,3,RevolvingUtilizationOfUnsecuredLines,0.1330,warn
2,3,age,0.1212,warn
3,4,RevolvingUtilizationOfUnsecuredLines,0.2329,warn
4,4,NumberOfOpenCreditLinesAndLoans,0.1289,warn
5,5,age,0.1355,warn
6,6,RevolvingUtilizationOfUnsecuredLines,0.1717,warn
7,6,age,0.1018,warn
8,7,RevolvingUtilizationOfUnsecuredLines,0.1767,warn
9,7,MonthlyIncome,0.1214,warn


## Report: Suggestions for Production-Scale Schema Drift Handling

### Introduction

The current simulation effectively demonstrates a data pipeline with batch ingestion, feature engineering, model training, and observability, including detection of schema drift. The `run_once` function handles schema drift by reading the entire `credit_risk_lake.csv` file, concatenating new batches (with `pd.concat` ensuring columns are unioned and new ones are added), and then overwriting the entire CSV. While functional for this simulation, this approach has limitations for large-scale production environments.



### 1. Limitations of Current CSV-based Approach

**Current Method**: The simulation's `run_once` function handles schema drift by:
1.  Reading the entire `LAKE_PATH` (a CSV file) into memory.
2.  Concatenating the new `batch_df` with the existing `lake_df` using `pd.concat(..., sort=False)`. This is crucial for handling schema evolution (e.g., adding `CreditScoreBand` at run #6), as it unions columns and ensures the combined DataFrame has all necessary columns.
3.  Rewriting the entire combined DataFrame back to `LAKE_PATH`.

**Limitations for Production**:
-   **Performance**: For very large datasets, reading the entire file into memory and rewriting it on every batch is highly inefficient and resource-intensive.
-   **Atomicity**: Overwriting the entire file is not an atomic operation, meaning failures during the write could lead to data corruption or loss.
-   **Scalability**: CSV is a row-oriented format that is not optimized for analytical queries or efficient appends, especially when schema changes occur.
-   **Lack of Versioning**: There's no inherent versioning or transactionality, making rollbacks or auditing difficult.
-   **Schema Enforcement**: While `pd.concat` handles the immediate alignment, CSV itself doesn't enforce schema, relying solely on `pandas` to interpret it correctly during reads.



### 2. Propose Robust File Formats: Parquet or Delta Lake

For production-scale pipelines, transitioning from CSV to more advanced file formats is critical. This directly addresses the performance and schema evolution challenges.

-   **Apache Parquet**:
    -   **Columnar Storage**: Stores data column by column, enabling highly efficient query performance as only necessary columns are read.
    -   **Compression**: Offers excellent compression, reducing storage costs and I/O.
    -   **Schema Evolution**: Supports schema evolution (adding new columns, renaming columns, type changes) more gracefully than CSV. While not fully ACID compliant, tools built on Parquet often handle schema evolution well.
    -   **Append-only**: Can be efficiently appended to by adding new Parquet files to a directory, rather than rewriting the entire dataset.

-   **Delta Lake / Apache Iceberg / Apache Hudi**:
    -   **ACID Transactions**: Provides Atomicity, Consistency, Isolation, Durability guarantees, which are crucial for data reliability in production.
    -   **Schema Enforcement & Evolution**: Offers explicit schema enforcement to prevent bad data and robust schema evolution capabilities, allowing controlled changes (e.g., adding non-nullable columns with defaults, dropping columns) without breaking existing pipelines.
    -   **Versioning (Time Travel)**: Allows accessing previous versions of data, enabling rollbacks, audits, and reproducibility.
    -   **Upserts/Deletes**: Supports efficient data modification operations, which are impossible with simple CSV appends.
    -   **Optimized Storage**: Built on Parquet, inheriting its columnar benefits, but adds a transactional layer.

**Recommendation**: For a production data lake with schema drift, **Delta Lake** (or a similar transactional storage layer) would be the most robust solution. It provides the best balance of performance, data integrity, and schema management features.

### 3. Introduce Schema Registry Concepts

A schema registry is a centralized store for managing and serving schemas, particularly useful in data streaming and large data environments. It decouples the schema from the data itself, allowing for independent evolution.

-   **Centralized Schema Management**: All data producers and consumers register and retrieve schemas from a single source.
-   **Schema Validation**: Incoming data can be validated against its registered schema, rejecting malformed data early in the pipeline.
-   **Compatibility Checks**: A schema registry can enforce compatibility rules (e.g., backward, forward, full compatibility) when new schema versions are registered. This ensures that changes won't break existing consumers or historical data.
-   **Data Contracts**: Formalizes the data contract between different services, improving data governance and reliability.

**Examples**: Confluent Schema Registry (often used with Apache Kafka and Avro/Protobuf), native schema features in cloud data warehouses (e.g., BigQuery's schema evolution capabilities), or custom solutions with tools like Apache Avro or Protobuf for serialization. The `EXPECTED_SCHEMA` and `diff_schema` in the current simulation mimic a very basic form of schema definition and drift detection; a schema registry externalizes and enhances this.

### 4. Outline Schema Evolution Strategies

When schemas change, pipelines need clear strategies to handle these changes gracefully. Compatibility modes define how strict these changes can be.

-   **Backward Compatibility**: New consumers can read old data. This is typically achieved by only adding new optional fields or removing fields that are no longer mandatory. The current `pd.concat(..., sort=False)` approach with default `NaN` fills for new columns implicitly provides a form of backward compatibility for existing features.

-   **Forward Compatibility**: Old consumers can read new data. This is harder to achieve and often means new fields must be optional or ignored by older consumers. It requires careful design and often versioning of clients.

-   **Full Compatibility**: Both backward and forward compatibility are maintained.

-   **Strategies for Column Changes**:
    -   **Adding Columns**: New columns should ideally be optional. If mandatory, they need default values for historical data or a specific migration strategy.
    -   **Removing Columns**: Consumers need to be updated to no longer expect these columns. Often, soft-deletes (marking as deprecated) or a phased rollout are preferred.
    -   **Renaming Columns**: Treat as a removal of the old column and an addition of a new one. Data transformations are needed to map old names to new names (like the `DebtRatio`/`DebtRatioPct` handling in `engineer_features`).
    -   **Changing Data Types**: This is the most disruptive. Often requires data migration and careful coordination between producers and consumers. Implicit casting (e.g., string to numeric) can work if data is clean, but explicit transformation is safer.

In a production system, these strategies would be enforced by the chosen file format (e.g., Delta Lake) and schema registry, preventing incompatible changes from being deployed.

### 5. Integrate with a Data Warehousing Solution

While a data lake (using formats like Delta Lake) provides raw storage and schema evolution, integrating with a managed data warehousing solution offers further benefits for structured data access and governance.

-   **Managed Services**: Cloud data warehouses like **Google BigQuery**, **Snowflake**, or **Databricks Lakehouse** provide fully managed infrastructure that inherently handles storage, compute, and often schema management at scale.
-   **Built-in Schema Evolution**: These platforms have robust, built-in mechanisms for schema evolution, allowing columns to be added, modes to be changed (e.g., NULLABLE to REQUIRED with default values), and even types to be relaxed (e.g., INT to FLOAT).
-   **ACID Compliance**: All these solutions offer strong transactional guarantees.
-   **Query Optimization**: They are highly optimized for analytical queries, making data access faster and more efficient for downstream applications (e.g., ML feature stores, dashboards).
-   **Unified Governance**: Centralized platform for access control, auditing, and metadata management.

**Recommendation**: Data from the initial raw data lake (e.g., Delta Lake) would be transformed and loaded into a structured table in BigQuery or a similar data warehouse. This provides a robust and performant serving layer for downstream ML models and analytics, with schema evolution handled by the platform itself.

### Conclusion: A Phased Approach to Robust Schema Drift Handling

To move from the current simulation's effective but inefficient CSV-based schema drift handling to a production-scale solution, a phased approach would involve:

1.  **Migrate to a Transactional Data Lake Format**: Adopt **Delta Lake** (or Apache Iceberg/Hudi) on top of cloud object storage (e.g., GCS, S3). This immediately addresses the performance, atomicity, and basic schema evolution issues, allowing efficient appends and controlled schema changes.

2.  **Implement a Schema Registry**: Introduce a **Schema Registry** (e.g., Confluent Schema Registry with Avro) to formally define, validate, and manage schemas. This creates a data contract and ensures compatibility, rejecting invalid data before it corrupts the lake.

3.  **Refine Schema Evolution Strategies**: Establish clear guidelines and automated processes for handling schema changes (additions, removals, type changes) based on backward/forward compatibility requirements, enforced by the schema registry and the chosen data lake format.

4.  **Leverage a Cloud Data Warehouse**: For serving data to ML models and analytics, load transformed and curated data from the Delta Lake into a **managed data warehouse like Google BigQuery**. This provides a highly scalable, performant, and schema-flexible serving layer with built-in governance.

By implementing these improvements, the pipeline can gracefully handle schema drift at production scale, ensuring data quality, pipeline stability, and efficient resource utilization.

# Task
Based on the provided credit risk pipeline simulation and the detailed report on production-scale schema drift handling, propose comprehensive production-grade architectures for each major component of the pipeline. This includes data ingestion and storage, schema management, feature engineering and model training, model serving and inference, observability and monitoring, and overall orchestration and MLOps. The architectures should address challenges related to scalability, reliability, fault tolerance, cost-efficiency, improved schema evolution handling, data governance, and MLOps capabilities, building upon the suggestions made in the schema drift report.

## Review Current Pipeline Components

### Subtask:
Summarize the existing components and their functions in the current Colab simulation, identifying key stages like batch generation, schema validation, data lake append, feature engineering, model training/reuse, and metric logging.


### Summary of Current Pipeline Components

The current credit risk pipeline simulation, as implemented in the Colab notebook, consists of several key components that orchestrate data processing, model management, and observability. Here's a breakdown:

*   **`generate_batch` Function**: This component is responsible for generating synthetic data batches for each run. It simulates new incoming data, including injecting schema drift (column rename, new column) and data drift (distribution shifts) at specific run numbers.

*   **Schema Definition and Validation (`EXPECTED_SCHEMA`, `snapshot_schema`, `diff_schema`, `SchemaDiff`)**: The `EXPECTED_SCHEMA` dictionary defines the anticipated schema for incoming data. The `snapshot_schema` function infers the data types of a given DataFrame, and `diff_schema` compares the observed schema of an incoming batch against the `EXPECTED_SCHEMA` to detect any schema drift (added, removed, or type-changed columns). This basic schema validation is performed on each batch.

*   **Metadata Store (`MetadataStore` Class)**: This class manages an SQLite database (`metadata.db`) to log all pipeline activities and metrics. It records run details, data lineage (tracking data movement between stages), schema events (drift occurrences), data drift metrics, model performance metrics, and fairness metrics. It provides `start_run`, `finish_run`, and various `log_*` methods for persistent record-keeping.

*   **Pipeline Orchestrator (`run_once` Function)**: This is the core function that orchestrates a single end-to-end run of the pipeline. For each batch, it performs the following sequential steps:
    1.  **Ingestion**: Generates a new batch using `generate_batch` and saves it as a CSV file.
    2.  **Schema Validation**: Detects schema drift using `diff_schema` against `EXPECTED_SCHEMA` and logs any changes.
    3.  **Data Lake Append**: Appends the new batch to a cumulative `credit_risk_lake.csv` file. This is handled by reading the entire existing lake, concatenating the new batch (using `pd.concat` with `sort=False` to handle new columns), and overwriting the lake CSV. This is the primary mechanism for data storage and schema evolution handling in the simulation.
    4.  **Data Drift Detection**: Compares the latest batch against the very first batch (`batch_000.csv`) using `compute_feature_drift` to calculate Population Stability Index (PSI) and KS-test statistics for numeric features, and Total Variation Distance (TVD) for categorical features, logging any warnings or alerts.

*   **Feature Engineering (`engineer_features` Function)**: This component takes the cumulative data lake, performs data cleaning (imputation for `MonthlyIncome` and `NumberOfDependents`, outlier treatment for `age`), and generates new features (e.g., `Age_Band`, `income_per_person`, `has_dependents`, `high_debt_ratio`, `total_past_due`). Notably, it includes logic to handle the renamed `DebtRatio` column (`DebtRatioPct`) for pipeline resilience.

*   **Model Training and Reuse**: Within `run_once`, a `LogisticRegression` model is periodically trained (every `RETRAIN_EVERY` runs, or if no model exists). Data is split into train/test sets, preprocessed using a `ColumnTransformer` (scaling numeric, one-hot encoding categorical), and the model is trained. If not retraining, the existing model is reused. The trained model and preprocessor are saved to `model.joblib`.

*   **Metric Logging and Observability**: After model training, performance metrics (AUC, Accuracy, F1-score) are calculated on the test set and logged. Fairness metrics (Demographic Parity Disparity) across sensitive groups (age bands) are also computed and logged using `fairlearn.metrics`. All these metrics, along with schema drift events and data drift results, feed into the `build_dashboard` function, which generates an HTML dashboard for visualization.

## Define Architectural Goals

### Subtask:
Establish the key objectives for the new architectures, such as scalability, reliability, fault tolerance, cost-efficiency, improved schema evolution handling, data governance, and MLOps capabilities, building on the previous discussion about production-scale improvements.


### Architectural Goals for the Production-Grade Pipeline

To address the limitations of the current simulation and build a robust credit risk pipeline, the following architectural goals are established:

1.  **Scalability**: The architecture must be able to handle increasing data volumes and processing demands efficiently, without significant performance degradation. This includes scaling data ingestion, storage, feature engineering, and model inference components independently.

2.  **Reliability & Fault Tolerance**: The pipeline should be resilient to failures at any stage. This means ensuring data integrity, providing mechanisms for retry logic, error handling, and minimizing downtime. Critical components should have built-in redundancy and automated recovery capabilities.

3.  **Cost-Efficiency**: The proposed solutions should optimize for cloud resource utilization, leveraging serverless or managed services where appropriate to reduce operational overhead and cost, while maintaining performance and reliability standards.

4.  **Improved Schema Evolution Handling**: Moving beyond simple `pd.concat` for schema changes, the new architecture must incorporate robust mechanisms for schema definition, validation, and evolution. This includes support for controlled schema changes (e.g., adding/removing columns, altering data types) with compatibility checks to prevent downstream pipeline breaks.

5.  **Data Governance**: The architecture should enable comprehensive data governance, encompassing:
    *   **Data Quality**: Implementing validation rules and anomaly detection to ensure data accuracy and consistency.
    *   **Data Lineage**: Maintaining clear records of data transformations and movements from source to destination.
    *   **Access Control**: Securely managing who can access and modify data at various stages.
    *   **Auditing**: Providing a clear audit trail of data changes and pipeline executions.

6.  **MLOps Capabilities**: The pipeline should be designed with MLOps principles in mind, facilitating:
    *   **Automation**: Automating data ingestion, processing, model training, deployment, and monitoring.
    *   **Reproducibility**: Ensuring that models and results can be reproduced consistently.
    *   **Versioning**: Managing versions of data, features, models, and code.
    *   **Continuous Integration/Continuous Delivery (CI/CD)**: Supporting automated testing and deployment of pipeline changes and model updates.
    *   **Monitoring & Alerting**: Implementing proactive monitoring for data drift, model performance degradation, and fairness issues, with automated alerts.

## Design High-Level Data Ingestion & Storage Architecture

### Subtask:
Propose a high-level architecture diagram for the data ingestion and data lake layers, incorporating cloud-native services (e.g., Cloud Storage, Event Streams) for raw data and transactional data lake formats (e.g., Delta Lake on GCS) for the curated data lake.


### 1. Cloud-Native Services for Data Ingestion and Primary Storage

For a production-grade credit risk pipeline on Google Cloud Platform (GCP), we propose the following services for data ingestion and primary storage:

*   **Data Ingestion Sources**:
    *   **Google Cloud Pub/Sub**: For real-time streaming data ingestion (e.g., application events, customer interactions, transactional data). Pub/Sub provides a highly scalable, durable, and flexible messaging service capable of handling millions of events per second. It's ideal for low-latency data arrival, enabling near real-time updates to credit risk profiles or immediate fraud detection.
    *   **Cloud Storage (e.g., GCS buckets)**: For batch data ingestion. This is suitable for receiving large volumes of structured or unstructured data files (e.g., CSV, JSON, Parquet) from external systems, on-premise databases, or other cloud providers at scheduled intervals. It offers high durability, availability, and scalability.

*   **Primary Raw Data Storage (Landing Zone)**:
    *   **Google Cloud Storage (GCS) Buckets**: GCS will serve as the primary landing zone for all incoming raw data, whether streamed via Pub/Sub (which can push to GCS) or uploaded directly as batch files. GCS provides object storage with extreme durability, high availability, and cost-effectiveness. It's schema-agnostic, allowing for the storage of any data format. Data in this layer will be immutable, serving as a single source of truth for raw events. This raw data forms the foundation for data lineage and recovery processes.

### 2. Transactional Data Lake (Curated Zone)

For the curated data lake, where data is prepared for feature engineering and model training, we propose a transactional storage layer on top of GCS:

*   **Delta Lake on Google Cloud Storage (GCS)**: Delta Lake provides an open-source storage layer that brings ACID (Atomicity, Consistency, Isolation, Durability) transactions, scalable metadata handling, and unified streaming and batch data processing to existing data lakes. When deployed on GCS, it offers:
    *   **ACID Transactions**: Ensures data integrity and consistency, preventing data corruption during writes or concurrent operations. This is a significant improvement over the current CSV-based approach, which lacks atomicity.
    *   **Schema Enforcement and Evolution**: This is a core benefit. Delta Lake allows for explicit schema definition and validates incoming data against it, rejecting invalid records. Critically, it supports controlled schema evolution (e.g., adding new columns, changing column types safely, dropping columns) without breaking downstream pipelines. This addresses the challenge of schema drift far more robustly than `pd.concat`.
    *   **Versioning (Time Travel)**: Delta Lake maintains a history of all changes made to the data, allowing users to query previous versions of the data. This enables rollbacks, audits, and reproducibility, aligning with the architectural goals for reliability and MLOps.
    *   **Performance Optimization**: Built on Parquet, Delta Lake inherits columnar storage benefits for efficient reads. It also offers optimizations like data skipping, Z-ordering, and compaction to further enhance query performance.

*   **Integration with Data Processing Engines**: Data stored in Delta Lake on GCS can be easily accessed and processed by various compute engines, such as:
    *   **Databricks (on GCP)**: Provides a fully managed platform with optimized Delta Lake runtime.
    *   **Google Cloud Dataproc**: For running Apache Spark, Flink, and Hadoop clusters that can interact with Delta Lake.
    *   **Google Cloud Dataflow**: For large-scale data processing pipelines (though native Delta Lake support might require custom connectors or Spark-on-Dataflow).
    *   **BigQuery (External Tables)**: BigQuery can query Delta Lake tables on GCS as external tables, providing a powerful analytical interface without moving data.

## Design Schema Management Architecture

### Subtask:
Detail the integration of a Schema Registry (e.g., Confluent Schema Registry or a cloud-native equivalent) within the ingestion pipeline to enforce, validate, and evolve schemas, ensuring data quality and compatibility across services.


### 1. Role and Benefits of a Schema Registry

A Schema Registry serves as a centralized metadata management system, crucial for robust data pipelines in a production MLOps environment. It stores, manages, and serves schemas for data exchanged between different services, ensuring data quality, compatibility, and efficient evolution.

**Role of a Schema Registry:**
*   **Centralized Schema Definition**: Provides a single source of truth for all data schemas, preventing inconsistencies and ambiguities across the organization.
*   **Schema Evolution Management**: Facilitates controlled evolution of schemas over time, allowing for additions, modifications, or deprecations of fields without breaking existing data producers or consumers.
*   **Schema Validation**: Enables validation of incoming data against its expected schema, catching malformed data early in the ingestion process.
*   **Data Contracts**: Establishes clear data contracts between different services, improving communication and reducing integration issues.
*   **Data Governance**: Supports data governance initiatives by providing discoverability of data structures, facilitating data lineage, and ensuring compliance.

**Benefits over the Current Simulation's Basic Schema Validation:**

The current simulation uses `EXPECTED_SCHEMA` and `diff_schema` to detect schema drift, which is a rudimentary form of schema validation. While it identifies changes, it lacks the sophistication required for a production environment:

*   **No Centralized Enforcement**: The `EXPECTED_SCHEMA` is hardcoded. In a production setting, schemas originate from various sources and need a central repository to be shared and enforced across multiple microservices or data pipelines.
*   **Limited Compatibility Checks**: The `diff_schema` only reports differences (added, removed, type-changed). A Schema Registry, however, can enforce strict compatibility rules (e.g., backward, forward, full compatibility) before a new schema version is even allowed to be registered, actively preventing breaking changes.
*   **No Versioning of Schemas**: The current system doesn't formally version `EXPECTED_SCHEMA`. A Schema Registry maintains a history of all schema versions, allowing for traceability, auditing, and easier rollback.
*   **No Serialization Integration**: The simulation processes CSVs, which are schema-less. Production systems often use binary serialization formats (like Avro or Protobuf) that embed schema information. A Schema Registry integrates tightly with these formats, managing the schema IDs and definitions necessary for efficient serialization/deserialization.
*   **Manual Handling of Schema Changes**: While `engineer_features` includes logic for renamed columns (`DebtRatio`/`DebtRatioPct`), this is a manual, reactive fix. A Schema Registry-driven approach would allow for more automated and proactive handling of such changes, potentially even through automated schema migration tools.

### 2. Proposed Schema Management Architecture Integration

Integrating a Schema Registry into the data ingestion pipeline fundamentally changes how schemas are defined, validated, and evolved. For a GCP-based credit risk pipeline, a practical approach involves:

**A. Schema Definition and Storage:**
*   **Schema Registry Service**: Use a dedicated Schema Registry. While Confluent Schema Registry is a popular open-source option, for GCP, a cloud-native equivalent or a managed service that offers similar functionality (e.g., integrating with Dataproc, Dataflow, or leveraging BigQuery's strong schema capabilities for the curated layer) would be preferred. This registry will store Avro or Protobuf schemas, which are self-describing and support robust schema evolution.
*   **Version Control for Schemas**: Schemas themselves (e.g., Avro `.avsc` files) should be stored in a version control system (e.g., Git). Any changes to a schema would follow a code review and approval process before being registered with the Schema Registry.

**B. Integration within the Ingestion Pipeline (Producers):**
1.  **Producer-Side Schema Registration**: Data producers (e.g., applications generating credit application data, transactional systems) would use client libraries that integrate with the Schema Registry. Before sending data, the producer would register its data's schema with the registry, obtaining a schema ID. If the schema is new or has evolved, the registry would perform compatibility checks (e.g., backward compatibility) against previous versions before allowing registration.
2.  **Data Serialization with Schema ID**: The producer serializes data using a format like Avro or Protobuf, embedding the schema ID in the message payload or as metadata. This ensures that each data record carries its schema context efficiently.
3.  **Data Ingestion to Raw Layer**: The serialized data (e.g., Avro-encoded messages via Pub/Sub, or Avro/Parquet files uploaded to GCS) is ingested into the raw landing zone (GCS buckets).

**C. Integration within the Processing Pipeline (Consumers/Data Lake Loading):**
1.  **Consumer-Side Schema Retrieval**: Data consumers (e.g., Dataflow/Spark jobs processing raw data from Pub/Sub or GCS) query the Schema Registry using the embedded schema ID to retrieve the exact schema used for serialization. This allows for correct deserialization.
2.  **Schema Validation and Enforcement**: As data is processed and prepared for the transactional data lake (Delta Lake on GCS), the processing job performs explicit schema validation against the *expected* schema for the Delta Lake table. This validation can be more stringent than the producer-side checks. Records failing validation can be routed to a dead-letter queue or error table for investigation.
3.  **Controlled Schema Evolution for Delta Lake**: When a new schema version is approved and registered, the Delta Lake table's schema can be *evolved* in a controlled manner. Delta Lake's `mergeSchema` or `overwriteSchema` options, combined with schema validation, enable safe additions of columns, reordering, or even type changes according to predefined compatibility rules. This is a significant improvement over `pd.concat`, which simply unions columns without explicit validation.

**D. Schema Drift Handling:**
*   **Proactive Prevention**: By enforcing compatibility rules at schema registration and validating data at ingestion, many breaking schema drifts are prevented *before* they reach the data lake.
*   **Graceful Evolution**: For allowed changes (e.g., adding an optional column), the pipeline can evolve gracefully. Older data will have `null` for new columns, while new data will populate them. Renaming columns would be handled as a transformation step (similar to the current `DebtRatio` logic, but driven by registered schema changes) within the data processing job before writing to Delta Lake.
*   **Alerting**: The Schema Registry or data processing jobs can trigger alerts if incompatible schema changes are attempted or if data fails validation against its expected schema.

## Design Feature Engineering & Model Training Architecture

### Subtask:
Outline an architecture for scalable feature engineering and model training, leveraging managed services (e.g., Vertex AI Workbench, Databricks ML Runtime) and ensuring reproducibility and versioning of features and models.


### 1. Feature Engineering with a Feature Store

The current simulation performs feature engineering on the full cumulative data lake using a function (`engineer_features`) that includes data cleaning, outlier treatment, and creation of derived features. In a production-grade architecture, this process needs to be robust, scalable, and ensure consistency across training and inference.

**Proposed Feature Engineering Process:**

1.  **Data Source**: Cleaned and validated data from the **Transactional Data Lake (Delta Lake on GCS)** serves as the primary input for feature engineering. This ensures that the input data is reliable and has undergone initial schema validation and evolution handling.

2.  **Feature Transformation Logic**: The feature engineering logic, as exemplified by `engineer_features`, will be implemented using distributed processing frameworks capable of handling large datasets. For GCP, this could be:
    *   **Google Cloud Dataflow (Apache Beam)**: Ideal for stream and batch processing, providing unified pipelines for transformations.
    *   **Google Cloud Dataproc (Apache Spark)**: A managed Spark service, suitable for complex transformations on large data volumes.
    *   **Databricks (on GCP)**: Offers optimized Spark runtime and native integration with Delta Lake, making it a strong contender for feature computation.

3.  **Output to Feature Store**: Instead of just using the engineered features temporarily for model training, the results of the feature engineering process will be stored in a **Feature Store**.

**Introduction to a Feature Store:**
A Feature Store is a centralized repository that standardizes the definition, storage, and access of machine learning features. It provides a consistent interface to serve features for both model training (historical features) and online inference (latest features), solving the critical problem of "training-serving skew" (discrepancies between features used during training and those used during serving).

**Benefits of a Feature Store:**

*   **Reproducibility**: By versioning features and the code used to generate them, a Feature Store ensures that the exact same features can be reproduced for model training, retraining, and specific inference requests. This is crucial for debugging and auditing.
*   **Consistency (Training-Serving Skew Prevention)**: It guarantees that the features used during model training are identical to those used during model inference. This is achieved by storing precomputed feature values and providing a unified API for retrieval in both scenarios.
*   **Reusability**: Features can be defined once and reused across multiple models and teams, reducing redundant development effort and promoting standardization. For example, `Age_Band` or `income_per_person` could be used by various credit risk models.
*   **Scalability**: Production-grade Feature Stores (e.g., Vertex AI Feature Store, Feast) are designed to handle high-throughput, low-latency requests for online inference and large-scale batch retrievals for training.
*   **Data Governance & Discovery**: They provide a central catalog of available features, improving discovery and enabling better governance, access control, and lineage tracking for ML data assets.
*   **Timeliness**: Feature Stores can manage the freshness of features, ensuring that models always get the most up-to-date data for inference while maintaining consistency for historical training data.

**Implementation on GCP (Example):**

*   **Vertex AI Feature Store**: This managed service on GCP would be an ideal choice. It allows defining feature views, ingesting batch features (e.g., from Dataflow/Dataproc outputting to BigQuery or GCS), and serving them for online predictions (low-latency lookup) and batch predictions (large-scale retrieval).
*   **Offline Store**: BigQuery is often used as the offline store within Vertex AI Feature Store for historical feature data used in training.
*   **Online Store**: Managed low-latency databases (e.g., Bigtable, Redis) are used for online feature serving.

### 2. Model Training and Versioning

The simulation periodically retrains a `LogisticRegression` model, saving the model and preprocessor to `model.joblib`. In a production setting, model training needs to be scalable, reproducible, and seamlessly integrated into an MLOps workflow.

**Proposed Model Training Process:**

1.  **Training Data Acquisition**: Training data (features and target) will be retrieved from the **Feature Store**.
    *   For batch training, an offline store (like BigQuery or GCS with Parquet/Delta Lake) connected to the Feature Store will provide historical feature values. This ensures that features used for training are consistent with those served for inference.

2.  **Managed Training Environment**: Instead of running training scripts in a local Colab environment, a managed service for model training will be used to provide scalable compute resources and integrated MLOps capabilities.
    *   **Vertex AI Training**: Offers fully managed custom training with various machine types, GPU acceleration, and integration with other Vertex AI services (e.g., Feature Store, Model Registry). It supports containerized training jobs, ensuring environment consistency and reproducibility.
    *   **Databricks ML Runtime (on GCP)**: Provides an optimized Spark-based environment for large-scale ML, with built-in MLflow for experiment tracking and model management.

3.  **Experiment Tracking and Model Versioning (MLflow/Vertex AI Metadata)**:
    *   **Experiment Tracking**: During training, all relevant metadata (hyperparameters, training data version, metrics like AUC, Accuracy, F1, fairness metrics) will be logged using a dedicated experiment tracking system.
        *   **Vertex AI Metadata**: Provides a centralized repository for tracking ML metadata, including executions, artifacts, and parameters.
        *   **MLflow**: An open-source platform for managing the ML lifecycle, including experiment tracking, project packaging, and model management. It can be integrated with Databricks or run independently on GCP.
    *   **Model Versioning**: Once a model is trained and evaluated, it will be registered with a **Model Registry**.
        *   **Vertex AI Model Registry**: A centralized repository to manage and version ML models. It allows storing model artifacts, metadata, and stage transitions (e.g., staging, production).
        *   **MLflow Model Registry**: Similar functionality within the MLflow ecosystem.
    *   This ensures that the preprocessor (`ColumnTransformer`) and the model (`LogisticRegression`) are versioned together as a single artifact, preventing discrepancies between feature preprocessing and model expectations.

4.  **Automated Retraining**: The periodic retraining logic (every `RETRAIN_EVERY` runs in the simulation) will be orchestrated by a workflow manager (e.g., Vertex AI Pipelines, Cloud Composer). Retraining can be triggered based on a schedule, data drift detection, or model performance degradation alerts.

5.  **Reproducibility**: To ensure reproducibility, the training process will leverage:
    *   **Containerization**: Training code and dependencies are packaged into Docker containers (e.g., used by Vertex AI Training), ensuring consistent environments.
    *   **Code Version Control**: Training scripts, feature engineering code, and hyperparameter configurations are managed in a Git repository.
    *   **Data Versioning (via Feature Store)**: The Feature Store provides access to specific versions of historical feature data, guaranteeing that the exact data used for training can be retrieved.

## Design Model Serving & Inference Architecture

### Subtask:
Propose an architecture for serving the trained credit risk model for inference, focusing on low-latency predictions using managed endpoints (e.g., Vertex AI Endpoints) and considering batch inference options for periodic scoring.


### 1. Requirements for Model Serving

To effectively serve the credit risk model in a production environment, both online (real-time) and batch (offline) inference capabilities are essential, each with distinct requirements:

**A. Online Inference (Real-time Predictions)**

*   **Low Latency**: Critical for applications requiring immediate credit risk assessments (e.g., instant loan approvals, real-time fraud detection). Predictions must be returned within milliseconds.
*   **High Availability**: The inference service must be continuously available to respond to requests, often requiring auto-scaling and redundancy to handle fluctuating traffic loads.
*   **High Throughput**: Ability to process a large number of concurrent prediction requests efficiently.
*   **Scalability**: Automatically scale compute resources up or down based on demand to maintain performance and optimize costs.
*   **Monitoring**: Real-time monitoring of service health, prediction latency, error rates, and model performance metrics is crucial.
*   **A/B Testing & Canary Deployments**: Support for deploying multiple model versions simultaneously to compare performance and safely roll out new models.
*   **Feature Consistency**: Ensure that features used for online inference are computed and retrieved in the exact same way as during training, preventing training-serving skew.

**B. Batch Inference (Offline Scoring)**

*   **High Throughput**: Ability to score large datasets (e.g., an entire portfolio of customers) efficiently, potentially processing millions of records.
*   **Cost-Efficiency**: Optimized for batch processing, minimizing compute costs for large-scale, non-time-sensitive tasks.
*   **Scalability**: Process datasets of varying sizes, often leveraging distributed processing frameworks.
*   **Data Consistency**: Ensure that the data used for batch inference is consistent and up-to-date, typically sourced from data lakes or warehouses.
*   **Auditability & Reproducibility**: Results must be auditable, and the process should be reproducible for regulatory compliance and debugging.
*   **Integration with Downstream Systems**: Seamless delivery of batch predictions to other data systems (e.g., data warehouses, reporting tools).

### 2. Online Inference Architecture (Real-time Predictions)

For real-time credit risk assessments, the online inference architecture must deliver low-latency predictions and maintain high availability and scalability. This setup ensures that applications can query the model synchronously for immediate decisions.

**Key Components:**

1.  **Request Ingestion (API Gateway)**:
    *   **Cloud Load Balancing / API Gateway**: Incoming prediction requests from client applications (e.g., loan application portals, fraud detection systems) will first hit a load balancer or an API Gateway (like **Apigee API Management** or **Cloud Endpoints**). This provides a single entry point, handles traffic management, authentication, and basic request validation.

2.  **Online Feature Retrieval (Feature Store)**:
    *   **Vertex AI Feature Store (Online Store)**: Before making a prediction, the inference service needs to retrieve the latest features for the incoming entity (e.g., customer ID). The request will query the online serving layer of **Vertex AI Feature Store**. This layer, typically backed by a low-latency database like **Bigtable** or **Redis**, will provide precomputed feature values. This ensures feature consistency with training data and fast retrieval, preventing training-serving skew.

3.  **Prediction Service (Managed Endpoint)**:
    *   **Vertex AI Prediction (Managed Endpoints)**: The core of the online inference will be hosted on **Vertex AI Prediction Endpoints**. This managed service allows deploying models from the Vertex AI Model Registry, offering:
        *   **Scalability**: Automatic scaling (auto-scaling) of resources based on demand, ensuring performance under varying load.
        *   **High Availability**: Built-in redundancy and health checks for continuous availability.
        *   **Low Latency**: Optimized for fast prediction response times.
        *   **A/B Testing & Canary Deployments**: Support for deploying multiple model versions to the same endpoint, enabling traffic splitting for A/B testing or gradual rollout (canary deployments) of new models. This facilitates safe experimentation and model updates.
        *   **Integrated Monitoring**: Automatic integration with Vertex AI Model Monitoring for data drift, feature attribution drift, and model performance metrics.

4.  **Logging and Monitoring (Observability Integration)**:
    *   **Cloud Logging**: All inference requests and responses, along with any errors, will be logged to **Cloud Logging**. This provides a centralized system for debugging and auditing.
    *   **Cloud Monitoring**: Key metrics from the prediction endpoint (e.g., request latency, error rates, resource utilization) will be collected by **Cloud Monitoring**. Custom dashboards and alerts can be configured to proactively detect issues.

**Data Flow:**

Client Application Request → API Gateway → Vertex AI Feature Store (Online Retrieval) → Vertex AI Prediction Endpoint (Model Inference) → Client Application Response.


### 3. Batch Inference Architecture (Offline Scoring)

For periodic scoring of large datasets or non-time-sensitive use cases, batch inference is crucial. This architecture prioritizes cost-efficiency and high throughput over low-latency.

**Key Components:**

1.  **Data Source (Batch Feature Retrieval)**:
    *   **Vertex AI Feature Store (Offline Store) or BigQuery/Delta Lake**: Batch inference will retrieve features from the offline store of **Vertex AI Feature Store**, which is typically backed by **BigQuery** or **Delta Lake on GCS**. This ensures consistency with the features used during training and allows for scalable retrieval of historical feature sets.

2.  **Batch Prediction Service (Managed Batch Processing)**:
    *   **Vertex AI Batch Prediction**: This managed service is designed for high-throughput, asynchronous scoring of large datasets. Models from the Vertex AI Model Registry can be deployed for batch predictions. Vertex AI Batch Prediction handles:
        *   **Scalability**: Automatically scales compute resources to process large input datasets efficiently.
        *   **Cost-Efficiency**: Optimizes resource usage for batch jobs, allowing for cost-effective scoring of millions of records.
        *   **Integration**: Seamlessly integrates with GCS for input and output data.
    *   **Cloud Dataflow (Apache Beam)**: For more complex batch scoring scenarios that require custom pre-processing logic or integration with other data sources, **Cloud Dataflow** can be used. It allows running custom batch pipelines that read features, apply the model, and write predictions.
    *   **Dataproc (Apache Spark)**: For Spark-native batch scoring, especially when data resides in Delta Lake and complex Spark transformations are involved, **Cloud Dataproc** can be utilized.

3.  **Output Storage**: The results of batch predictions will be stored in a destination optimized for analytical queries and downstream consumption.
    *   **BigQuery**: Ideal for storing large volumes of batch predictions, allowing for easy querying, reporting, and integration with data warehousing and business intelligence tools.
    *   **Delta Lake on GCS**: Predictions can also be stored back into the Delta Lake for further analysis or as an input to other ML models.
    *   **Cloud Storage (GCS)**: Predictions can be written as Parquet or CSV files to GCS for direct consumption by other applications or for archival purposes.

4.  **Orchestration and Scheduling (Integration with Orchestration Layer)**:
    *   Batch inference jobs will be scheduled and orchestrated by the overall MLOps orchestration layer (e.g., **Cloud Composer/Airflow**, **Vertex AI Pipelines**). This ensures that batch predictions run at predefined intervals or are triggered by data arrival, and that the entire process is automated.

**Data Flow:**

Orchestration Trigger \u2192 Vertex AI Feature Store (Offline Retrieval) / BigQuery/Delta Lake \u2192 Vertex AI Batch Prediction / Cloud Dataflow / Dataproc (Model Inference) \u2192 BigQuery / Delta Lake / GCS (Prediction Output).

## Design Observability & Monitoring Architecture

### Subtask:
Architect a comprehensive observability solution, integrating monitoring for data drift, model performance, and fairness metrics. Include components for logging, alerting, and dashboarding (e.g., using Vertex AI Model Monitoring, Grafana, custom dashboards).


### 1. Key Aspects to Monitor

To ensure the health, reliability, and performance of the credit risk pipeline in a production environment, a comprehensive monitoring strategy must cover several critical areas:

**A. Data Quality Monitoring:**
This ensures the integrity and consistency of the data flowing through the pipeline, from raw ingestion to the curated feature store.

*   **Schema Drift**: Monitoring for unexpected changes in the structure (columns added, removed, or type-changed) of incoming data. This directly addresses the challenges highlighted in the schema drift report.
    *   *Metrics*: Number of schema changes (additions, removals, type changes), frequency of schema changes, compatibility level violations.
*   **Data Drift**: Monitoring for changes in the statistical properties or distributions of features over time, which can indicate shifts in the underlying data population or issues upstream.
    *   *Metrics*: Population Stability Index (PSI), Kolmogorov-Smirnov (KS) test p-values, Total Variation Distance (TVD) for categorical features, feature distributions (histograms), missing value rates, cardinality changes.
*   **Data Validation**: Monitoring for data that does not conform to predefined rules or constraints.
    *   *Metrics*: Count of invalid records, percentage of valid records, custom data quality checks (e.g., age within range, income non-negative).
*   **Data Freshness & Volume**: Ensuring data arrives on time and in expected quantities.
    *   *Metrics*: Time since last update, total rows ingested, batch processing latency.

**B. Model Performance Monitoring:**
This tracks how well the deployed model is performing on live data and identifies any degradation over time.

*   **Prediction Performance**: Tracking standard machine learning metrics against ground truth (when available).
    *   *Metrics*: AUC (Area Under the Curve), Accuracy, F1-score, Precision, Recall, Log Loss.
*   **Prediction Drift**: Monitoring for changes in the distribution of model predictions, which can indicate either data drift affecting model output or a shift in the model's behavior.
    *   *Metrics*: Distribution of predicted probabilities/scores, changes in predicted class balance.
*   **Feature Attribution Drift**: For explainable AI, monitoring how feature importances change over time, which can highlight shifts in what the model is focusing on.
    *   *Metrics*: SHAP values, LIME explanations, or other feature importance scores over time.
*   **Model Latency & Throughput**: Monitoring the operational efficiency of the model serving endpoint.
    *   *Metrics*: Prediction latency (P50, P90, P99), requests per second, error rates.

**C. Fairness Metrics Monitoring:**
This ensures that the model's predictions remain equitable across different sensitive groups, addressing potential biases.

*   **Demographic Parity Disparity**: Monitoring for differences in the positive prediction rate across different demographic groups (e.g., age bands, gender, income groups).
    *   *Metrics*: Demographic Parity Difference, Equal Opportunity Difference, Predictive Equality Difference for sensitive attributes (e.g., `Age_Band` as seen in the simulation).
*   **Group Performance Discrepancies**: Monitoring standard performance metrics (accuracy, F1) disaggregated by sensitive groups.
    *   *Metrics*: Accuracy by Age Band, F1-score by income level.

**D. System Health & Resource Utilization:**
Beyond ML-specific metrics, standard infrastructure monitoring is crucial.

*   *Metrics*: CPU utilization, Memory usage, Disk I/O, Network I/O, Uptime, Error rates, Queue sizes (for streaming components like Pub/Sub).

### 2. Architectural Components for Observability and Monitoring

A robust observability and monitoring architecture for the credit risk pipeline will involve several integrated components to collect, process, analyze, and visualize metrics, as well as trigger alerts for anomalies.

**A. Logging and Metrics Collection:**

1.  **Google Cloud Logging**: All pipeline components (data ingestion, feature engineering, model training, model serving, orchestration) will emit structured logs to Cloud Logging. This includes application logs, errors, warnings, and custom event logs (e.g., successful batch ingestion, model training completion).
2.  **Google Cloud Monitoring (formerly Stackdriver Monitoring)**: This service will be used to collect infrastructure metrics (CPU, memory, disk I/O of VMs, container resources), custom application metrics (e.g., number of records processed, feature engineering duration), and service-specific metrics (e.g., Pub/Sub message counts, Dataflow job status, Vertex AI endpoint latency).
3.  **Vertex AI Model Monitoring**: This specialized service is crucial for ML-specific observability. It automatically monitors deployed models for:
    *   **Data Drift**: Detects changes in input feature distributions between training data and serving data.
    *   **Feature Attribution Drift**: Monitors for changes in feature importance (e.g., using SHAP values), indicating a shift in how the model is making predictions.
    *   **Prediction Drift**: Tracks changes in the distribution of model predictions over time.
    It can be configured to integrate with Vertex AI Prediction Endpoints and BigQuery for batch predictions.
4.  **Custom Data Quality Checks**: For deeper data quality monitoring beyond schema and distribution drift, custom checks (e.g., constraint violations, value range checks) implemented within Dataflow or Dataproc jobs will push their results as custom metrics to Cloud Monitoring or store them in a dedicated BigQuery table.
5.  **Schema Registry Monitoring**: The Schema Registry itself will log events (e.g., new schema version registered, compatibility check failed) to Cloud Logging, and its operational metrics will be collected by Cloud Monitoring.

**B. Alerting System:**

1.  **Google Cloud Monitoring Alerting**: Based on the collected metrics, alerts will be configured in Cloud Monitoring. These alerts will trigger when:
    *   Data drift (e.g., PSI above threshold, KS p-value below threshold) is detected by Vertex AI Model Monitoring or custom checks.
    *   Model performance metrics (e.g., AUC, F1-score) drop below a predefined threshold.
    *   Fairness metrics (e.g., demographic parity disparity) exceed acceptable limits.
    *   System health metrics (CPU, memory, error rates) show anomalies.
    *   Pipeline job failures or prolonged processing times are observed.
2.  **Notification Channels**: Alerts will be routed to appropriate teams via configured notification channels (e.g., PagerDuty, Slack, email, SMS) to ensure timely intervention.

**C. Dashboarding and Visualization:**

1.  **Google Cloud Monitoring Dashboards**: Custom dashboards can be created within Cloud Monitoring to visualize all collected metrics in real-time, providing an operational overview of the pipeline's health and performance.
2.  **Looker / Google Data Studio (Looker Studio)**: For more detailed business-centric dashboards, especially for model performance, fairness, and data quality over longer periods, Looker or Data Studio can connect directly to BigQuery (where model and drift metrics are stored) to create rich, interactive visualizations. This would replace the static HTML dashboard in the simulation.
3.  **Vertex AI Model Monitoring Dashboards**: Vertex AI Model Monitoring provides out-of-the-box dashboards for visualizing data drift, prediction drift, and feature attribution drift for deployed models.

**D. Traceability and Data Lineage:**

1.  **Vertex AI Metadata**: Extends the concept of the `MetadataStore` in the simulation to a comprehensive, managed metadata store. It automatically tracks the lineage of ML artifacts (datasets, models, pipelines, executions), parameters, and metrics, providing an auditable trail of the entire ML lifecycle.
2.  **Custom Lineage (within Delta Lake)**: Delta Lake's transaction log inherently provides data versioning and can be leveraged to understand changes to the data lake tables over time. Integrations can be built to push this metadata to Vertex AI Metadata or a separate data catalog.

**Integration Summary:**

By integrating these components, the architecture moves from reactive detection (as in the simulation's `diff_schema` and dashboard) to proactive, automated monitoring and alerting, providing real-time visibility into data quality, model health, and system performance.

## Design Feature Engineering & Model Training Architecture

### Subtask:
Outline an architecture for scalable feature engineering and model training, leveraging managed services and ensuring reproducibility and versioning of features and models.


### 3. Model Training and Versioning

The simulation periodically retrains a `LogisticRegression` model, saving the model and preprocessor to `model.joblib`. In a production setting, model training needs to be scalable, reproducible, and seamlessly integrated into an MLOps workflow.

**Proposed Model Training Process:**

1.  **Training Data Acquisition**: Training data (features and target) will be retrieved from the **Feature Store**.
    *   For batch training, an offline store (like BigQuery or GCS with Parquet/Delta Lake) connected to the Feature Store will provide historical feature values. This ensures that features used for training are consistent with those served for inference.

2.  **Managed Training Environment**: Instead of running training scripts in a local Colab environment, a managed service for model training will be used to provide scalable compute resources and integrated MLOps capabilities.
    *   **Vertex AI Training**: Offers fully managed custom training with various machine types, GPU acceleration, and integration with other Vertex AI services (e.g., Feature Store, Model Registry). It supports containerized training jobs, ensuring environment consistency and reproducibility.
    *   **Databricks ML Runtime (on GCP)**: Provides an optimized Spark-based environment for large-scale ML, with built-in MLflow for experiment tracking and model management.

3.  **Experiment Tracking and Model Versioning (MLflow/Vertex AI Metadata)**:
    *   **Experiment Tracking**: During training, all relevant metadata (hyperparameters, training data version, metrics like AUC, Accuracy, F1, fairness metrics) will be logged using a dedicated experiment tracking system.
        *   **Vertex AI Metadata**: Provides a centralized repository for tracking ML metadata, including executions, artifacts, and parameters.
        *   **MLflow**: An open-source platform for managing the ML lifecycle, including experiment tracking, project packaging, and model management. It can be integrated with Databricks or run independently on GCP.
    *   **Model Versioning**: Once a model is trained and evaluated, it will be registered with a **Model Registry**.
        *   **Vertex AI Model Registry**: A centralized repository to manage and version ML models. It allows storing model artifacts, metadata, and stage transitions (e.g., staging, production).
        *   **MLflow Model Registry**: Similar functionality within the MLflow ecosystem.
    *   This ensures that the preprocessor (`ColumnTransformer`) and the model (`LogisticRegression`) are versioned together as a single artifact, preventing discrepancies between feature preprocessing and model expectations.

4.  **Automated Retraining**: The periodic retraining logic (every `RETRAIN_EVERY` runs in the simulation) will be orchestrated by a workflow manager (e.g., Vertex AI Pipelines, Cloud Composer). Retraining can be triggered based on a schedule, data drift detection, or model performance degradation alerts.

5.  **Reproducibility**: To ensure reproducibility, the training process will leverage:
    *   **Containerization**: Training code and dependencies are packaged into Docker containers (e.g., used by Vertex AI Training), ensuring consistent environments.
    *   **Code Version Control**: Training scripts, feature engineering code, and hyperparameter configurations are managed in a Git repository.
    *   **Data Versioning (via Feature Store)**: The Feature Store provides access to specific versions of historical feature data, guaranteeing that the exact data used for training can be retrieved.

## Design Orchestration & MLOps Architecture

### Subtask:
Describe an orchestration layer for the entire pipeline, using tools like Apache Airflow (or Cloud Composer), Kubeflow Pipelines, or Vertex AI Pipelines, to automate, schedule, and manage the end-to-end MLOps workflow.


### 1. The Need for a Robust Orchestration Layer

In the current simulation, the `run_once` function acts as a simple orchestrator, executing a predefined sequence of steps (data ingestion, schema validation, data lake append, data drift detection, feature engineering, model training/reuse, metric logging) in a linear fashion. While effective for demonstration, this approach has significant limitations for a production-grade MLOps pipeline:

*   **Monolithic Execution**: `run_once` is a single, long-running function. If any step fails, the entire run stops, making error recovery cumbersome and requiring manual intervention.
*   **Lack of Modularity and Reusability**: The logic for different stages is tightly coupled within one function, hindering independent development, testing, and reuse of individual components.
*   **Limited Scheduling and Triggering**: The simulation relies on manual execution or basic time-based triggering. Production pipelines require sophisticated scheduling (e.g., cron-based, event-driven, data arrival triggers) and the ability to define complex dependencies.
*   **No Dependency Management**: There's no explicit mechanism to manage dependencies between tasks, retries, or conditional execution paths based on the success/failure of upstream tasks.
*   **Scalability Challenges**: The `run_once` function runs within a single process. Production workloads demand distributed execution, allowing different stages to scale independently and leverage specialized compute resources.
*   **Limited Visibility and Control**: Debugging, monitoring progress, and gaining insight into the state of individual tasks within `run_once` is challenging. There's no built-in mechanism for pausing, resuming, or backfilling runs.
*   **No MLOps Workflow Integration**: It lacks inherent integration with MLOps best practices like CI/CD for pipeline code, automated model deployment, or comprehensive metadata tracking across the entire workflow.

For these reasons, a dedicated **orchestration layer** is indispensable for automating, managing, and monitoring the end-to-end MLOps workflow in a production environment. It transforms the pipeline from a script into a robust, observable, and maintainable system.

### 2. Proposed Orchestration Platforms

For a production-grade MLOps pipeline on Google Cloud Platform, we have several powerful options for orchestration, each with its strengths. We will focus on **Google Cloud Composer (Managed Apache Airflow)** and **Vertex AI Pipelines (Managed Kubeflow Pipelines)**.

**A. Google Cloud Composer (Managed Apache Airflow)**

Cloud Composer is a fully managed Apache Airflow service that enables you to author, schedule, and monitor pipelines programmatically. It's a highly flexible and widely adopted choice for orchestrating complex data workflows.

**Key Features and Benefits for MLOps:**
*   **DAGs (Directed Acyclic Graphs)**: Defines workflows as a series of tasks with clear dependencies, allowing for complex branching, parallel execution, and retries. Each stage of our credit risk pipeline (data ingestion, schema validation, feature engineering, model training, model deployment, monitoring) would be a task or a set of tasks within an Airflow DAG.
*   **Extensibility**: Airflow has a vast ecosystem of operators and sensors, including many for GCP services (e.g., `GCSOperator`, `BigQueryOperator`, `DataprocOperator`, `DataflowOperator`, `VertexAIOperator`). This makes it easy to integrate all the proposed architectural components.
*   **Scheduling and Triggering**: Supports cron-based scheduling, manual triggers, and can be integrated with event-driven triggers (e.g., Cloud Functions reacting to GCS file uploads can trigger Airflow DAGs). This addresses the need for sophisticated scheduling.
*   **Monitoring and Alerting**: Airflow's rich UI provides visibility into pipeline runs, task statuses, and logs. Cloud Composer integrates with Cloud Monitoring for operational metrics and Cloud Logging for detailed task logs, facilitating proactive alerting.
*   **Modularity**: Encourages breaking down monolithic processes into smaller, reusable tasks, improving development velocity and maintainability.
*   **Error Handling and Retries**: Built-in mechanisms for defining task retries, timeouts, and error callbacks, enhancing pipeline reliability.

**B. Vertex AI Pipelines (Managed Kubeflow Pipelines)**

Vertex AI Pipelines is a managed service for orchestrating machine learning workflows using Kubeflow Pipelines. It's specifically designed for MLOps and deeply integrated with the Vertex AI platform.

**Key Features and Benefits for MLOps:**
*   **ML-centric Design**: Purpose-built for ML workflows, providing native components for common ML tasks (data preprocessing, training, model evaluation, deployment) and seamless integration with other Vertex AI services (Feature Store, Model Registry, Prediction Endpoints).
*   **Component-based Approach**: Workflows are composed of reusable, containerized components, promoting modularity, reusability, and reproducibility. Each component runs in its own isolated environment.
*   **Metadata Tracking**: Automatically logs ML metadata (input/output artifacts, parameters, metrics) to Vertex AI Metadata, providing rich lineage and auditability of every pipeline run.
*   **Scalability**: Leverages Kubernetes for dynamic scaling of compute resources, making it suitable for large-scale ML experiments and production pipelines.
*   **Visualization**: Provides an intuitive UI to visualize pipeline graphs, component statuses, and experiment results, making debugging and monitoring easier.
*   **Conditional Execution and Parallelism**: Supports complex pipeline structures with conditional execution paths (e.g., retrain model only if data drift exceeds threshold) and parallel task execution.
*   **Managed Service**: Reduces operational overhead as Google manages the underlying infrastructure.

**Recommendation**: For an end-to-end MLOps pipeline where ML is a central concern, **Vertex AI Pipelines** offers superior integration with GCP's ML ecosystem and inherent ML metadata tracking. For broader data integration tasks, **Cloud Composer** remains a strong, flexible choice. A hybrid approach might involve Cloud Composer orchestrating the overall data movement and triggering Vertex AI Pipelines for the core ML lifecycle stages.

### 3. End-to-End MLOps Workflow Orchestration with Vertex AI Pipelines

Given the ML-centric nature of the credit risk pipeline, **Vertex AI Pipelines** is recommended as the primary orchestrator for the entire MLOps workflow. It provides a managed, scalable, and metadata-rich environment for building, deploying, and managing complex ML pipelines.

**A. Core Pipeline Stages and Integration with GCP Services:**

An end-to-end Vertex AI Pipeline for the credit risk model would look like this:

1.  **Data Ingestion & Landing (Trigger)**:
    *   **Trigger**: The pipeline run is initiated. This could be scheduled (e.g., daily/hourly for new batches), event-driven (e.g., via Cloud Functions triggered by new data landing in a GCS raw bucket or Pub/Sub topic), or manually invoked.
    *   **Component**: A pipeline component reads raw data from the GCS landing zone (or Pub/Sub via Dataflow/Spark streaming) and performs initial transformations.

2.  **Schema Validation & Data Lake Loading (Delta Lake)**:
    *   **Component**: A dedicated component (e.g., using Dataproc/Dataflow with Delta Lake libraries) performs advanced schema validation against the **Schema Registry**. It ensures data quality and compatibility.
    *   **Action**: Validated data is written to the **Transactional Data Lake (Delta Lake on GCS)**, handling schema evolution (e.g., adding new columns) in a controlled manner.
    *   **Metadata**: Vertex AI Metadata tracks the raw data artifact, the schema used, and the resulting Delta Lake table version.

3.  **Feature Engineering & Feature Store Population**:
    *   **Component**: A Dataflow or Dataproc job reads data from the Delta Lake, applies feature engineering logic (imputation, outlier treatment, new feature creation, similar to `engineer_features`).
    *   **Action**: The engineered features are then ingested into the **Vertex AI Feature Store** (offline store, typically BigQuery). This ensures feature consistency and reusability.
    *   **Metadata**: Vertex AI Metadata tracks the Delta Lake source, the feature engineering script/container, and the resulting features in the Feature Store.

4.  **Data Drift Detection & Monitoring**:
    *   **Component**: A component periodically samples new features from the Feature Store (or Delta Lake) and compares their distributions against a baseline (e.g., training data reference).
    *   **Action**: It computes data drift metrics (PSI, KS-test, TVD) for various features.
    *   **Conditional Execution**: Based on the severity of data drift, the pipeline can conditionally proceed to retraining or trigger alerts to human operators. (e.g., `if data_drift_severity == 'alert': send_email()`)
    *   **Metadata & Monitoring**: Drift metrics are logged to **Vertex AI Metadata** and pushed to **Cloud Monitoring** for alerting and **Vertex AI Model Monitoring** for visualization.

5.  **Model Training & Evaluation**:
    *   **Component**: A Vertex AI Training job (running a custom container with the model training code) is triggered.
    *   **Action**: It retrieves historical features and target labels from the **Vertex AI Feature Store** (offline store), trains the model (e.g., `LogisticRegression`), and evaluates its performance (AUC, Accuracy, F1) on a held-out test set. Fairness metrics are also calculated.
    *   **Conditional Execution**: If model performance or fairness metrics fall below predefined thresholds (e.g., compared to the previous model), the pipeline can decide not to deploy the new model or to trigger human review.
    *   **Metadata & Tracking**: Training parameters, data splits, model artifacts, and all evaluation metrics (performance, fairness) are logged to **Vertex AI Metadata** and **Vertex AI Experiments** for comprehensive tracking and comparison. The model artifact is stored in **Vertex AI Model Registry**.

6.  **Model Deployment & Endpoint Update (Online Inference)**:
    *   **Component**: If evaluation criteria are met, a deployment component (e.g., a custom `Endpoint` component) is invoked.
    *   **Action**: The newly trained model from the **Vertex AI Model Registry** is deployed to a **Vertex AI Prediction Endpoint**. This can involve traffic splitting for canary deployments or directly updating the model behind the endpoint.
    *   **Metadata**: Vertex AI Metadata tracks the deployed model version and the endpoint it's serving.
    *   **Monitoring Integration**: The new deployment is automatically integrated with **Vertex AI Model Monitoring** for continuous monitoring of data drift, prediction drift, and potential feature attribution drift in live traffic.

7.  **Batch Inference & Output Storage (Offline Scoring)**:
    *   **Component**: A **Vertex AI Batch Prediction** job (or a Dataflow/Dataproc job) is triggered.
    *   **Action**: It reads a large dataset of features (e.g., from Feature Store offline or Delta Lake) and generates predictions. The predictions are then written to **BigQuery** or back to the **Delta Lake** for downstream analytics or reporting.
    *   **Metadata**: Vertex AI Metadata tracks the batch inference job, the model version used, and the prediction output artifact.

8.  **Observability & Alerting**:
    *   **Integration**: All stages push logs to **Cloud Logging** and metrics to **Cloud Monitoring**. Alerts are configured in Cloud Monitoring based on custom metrics (e.g., schema validation failures, data drift warnings/alerts, model performance drops, fairness metric violations, pipeline component failures).
    *   **Dashboards**: **Cloud Monitoring Dashboards**, **Vertex AI Model Monitoring Dashboards**, and **Looker Studio** provide comprehensive visualization of pipeline health, data quality, and model performance.

**B. Benefits of Vertex AI Pipelines for Orchestration:**

*   **Managed & Scalable**: Reduces operational overhead and automatically scales resources.
*   **ML-Centric Components**: Native integration with Vertex AI services, providing purpose-built components for common ML tasks.
*   **Strong Metadata Tracking**: Automatically logs artifacts, parameters, and metrics to Vertex AI Metadata, providing end-to-end lineage and auditability.
*   **Reproducibility**: Containerized components and explicit artifact tracking ensure runs can be reproduced.
*   **Conditional Logic**: Allows for dynamic pipeline execution based on data quality checks, model performance, or drift detection results.
*   **Unified Platform**: Centralizes various MLOps functionalities within a single ecosystem.

## Final Task

### Subtask:
Provide a comprehensive summary of the proposed architectures, highlighting how they address the challenges of production-scale data pipelines and machine learning, and how they build upon the concepts discussed in the schema drift report.


# Task
Based on the provided credit risk pipeline simulation and the detailed report on production-scale schema drift handling, propose comprehensive production-grade architectures for each major component of the pipeline. This includes data ingestion and storage, schema management, feature engineering and model training, model serving and inference, observability and monitoring, and overall orchestration and MLOps. The architectures should address challenges related to scalability, reliability, fault tolerance, cost-efficiency, improved schema evolution handling, data governance, and MLOps capabilities, building upon the suggestions made in the schema drift report.

## Design Orchestration & MLOps Architecture

### Subtask:
Describe an orchestration layer for the entire pipeline, using tools like Apache Airflow (or Cloud Composer), Kubeflow Pipelines, or Vertex AI Pipelines, to automate, schedule, and manage the end-to-end MLOps workflow.


### 3. End-to-End MLOps Workflow Orchestration with Vertex AI Pipelines

Given the ML-centric nature of the credit risk pipeline, **Vertex AI Pipelines** is recommended as the primary orchestrator for the entire MLOps workflow. It provides a managed, scalable, and metadata-rich environment for building, deploying, and managing complex ML pipelines.

**A. Core Pipeline Stages and Integration with GCP Services:**

An end-to-end Vertex AI Pipeline for the credit risk model would look like this:

1.  **Data Ingestion & Landing (Trigger)**:
    *   **Trigger**: The pipeline run is initiated. This could be scheduled (e.g., daily/hourly for new batches), event-driven (e.g., via Cloud Functions triggered by new data landing in a GCS raw bucket or Pub/Sub topic), or manually invoked.
    *   **Component**: A pipeline component reads raw data from the GCS landing zone (or Pub/Sub via Dataflow/Spark streaming) and performs initial transformations.

2.  **Schema Validation & Data Lake Loading (Delta Lake)**:
    *   **Component**: A dedicated component (e.g., using Dataproc/Dataflow with Delta Lake libraries) performs advanced schema validation against the **Schema Registry**. It ensures data quality and compatibility.
    *   **Action**: Validated data is written to the **Transactional Data Lake (Delta Lake on GCS)**, handling schema evolution (e.g., adding new columns) in a controlled manner.
    *   **Metadata**: Vertex AI Metadata tracks the raw data artifact, the schema used, and the resulting Delta Lake table version.

3.  **Feature Engineering & Feature Store Population**:
    *   **Component**: A Dataflow or Dataproc job reads data from the Delta Lake, applies feature engineering logic (imputation, outlier treatment, new feature creation, similar to `engineer_features`).
    *   **Action**: The engineered features are then ingested into the **Vertex AI Feature Store** (offline store, typically BigQuery). This ensures feature consistency and reusability.
    *   **Metadata**: Vertex AI Metadata tracks the Delta Lake source, the feature engineering script/container, and the resulting features in the Feature Store.

4.  **Data Drift Detection & Monitoring**:
    *   **Component**: A component periodically samples new features from the Feature Store (or Delta Lake) and compares their distributions against a baseline (e.g., training data reference).
    *   **Action**: It computes data drift metrics (PSI, KS-test, TVD) for various features.
    *   **Conditional Execution**: Based on the severity of data drift, the pipeline can conditionally proceed to retraining or trigger alerts to human operators. (e.g., `if data_drift_severity == 'alert': send_email())
    *   **Metadata & Monitoring**: Drift metrics are logged to **Vertex AI Metadata** and pushed to **Cloud Monitoring** for alerting and **Vertex AI Model Monitoring** for visualization.

5.  **Model Training & Evaluation**:
    *   **Component**: A Vertex AI Training job (running a custom container with the model training code) is triggered.
    *   **Action**: It retrieves historical features and target labels from the **Vertex AI Feature Store** (offline store), trains the model (e.g., `LogisticRegression`), and evaluates its performance (AUC, Accuracy, F1) on a held-out test set. Fairness metrics are also calculated.
    *   **Conditional Execution**: If model performance or fairness metrics fall below predefined thresholds (e.g., compared to the previous model), the pipeline can decide not to deploy the new model or to trigger human review.
    *   **Metadata & Tracking**: Training parameters, data splits, model artifacts, and all evaluation metrics (performance, fairness) are logged to **Vertex AI Metadata** and **Vertex AI Experiments** for comprehensive tracking and comparison. The model artifact is stored in **Vertex AI Model Registry**.

6.  **Model Deployment & Endpoint Update (Online Inference)**:
    *   **Component**: If evaluation criteria are met, a deployment component (e.g., a custom `Endpoint` component) is invoked.
    *   **Action**: The newly trained model from the **Vertex AI Model Registry** is deployed to a **Vertex AI Prediction Endpoint**. This can involve traffic splitting for canary deployments or directly updating the model behind the endpoint.
    *   **Metadata**: Vertex AI Metadata tracks the deployed model version and the endpoint it's serving.
    *   **Monitoring Integration**: The new deployment is automatically integrated with **Vertex AI Model Monitoring** for continuous monitoring of data drift, prediction drift, and potential feature attribution drift in live traffic.

7.  **Batch Inference & Output Storage (Offline Scoring)**:
    *   **Component**: A **Vertex AI Batch Prediction** job (or a Dataflow/Dataproc job) is triggered.
    *   **Action**: It reads a large dataset of features (e.g., from Feature Store offline or Delta Lake) and generates predictions. The predictions are then written to **BigQuery** or back to the **Delta Lake** for downstream analytics or reporting.
    *   **Metadata**: Vertex AI Metadata tracks the batch inference job, the model version used, and the prediction output artifact.

8.  **Observability & Alerting**:
    *   **Integration**: All stages push logs to **Cloud Logging** and metrics to **Cloud Monitoring**. Alerts are configured in Cloud Monitoring based on custom metrics (e.g., schema validation failures, data drift warnings/alerts, model performance drops, fairness metric violations, pipeline component failures).
    *   **Dashboards**: **Cloud Monitoring Dashboards**, **Vertex AI Model Monitoring Dashboards**, and **Looker Studio** provide comprehensive visualization of pipeline health, data quality, and model performance.

**B. Benefits of Vertex AI Pipelines for Orchestration:**

*   **Managed & Scalable**: Reduces operational overhead and automatically scales resources.
*   **ML-Centric Components**: Native integration with Vertex AI services, providing purpose-built components for common ML tasks.
*   **Strong Metadata Tracking**: Automatically logs artifacts, parameters, and metrics to Vertex AI Metadata, providing end-to-end lineage and auditability.
*   **Reproducibility**: Containerized components and explicit artifact tracking ensure runs can be reproduced.
*   **Conditional Logic**: Allows for dynamic pipeline execution based on data quality checks, model performance, or drift detection results.
*   **Unified Platform**: Centralizes various MLOps functionalities within a single ecosystem.